[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/optimization/07_linear_quadratic_conic_programs/exercises.ipynb)

# Module 07 — Linear, Quadratic, and Conic Programs

Exercises: twenty fully solved problems in four tiers — concept checks, foundations, applications, and challenge proofs. Every numeric answer below is recomputed by a code cell that ran.

Setup for every verification cell below. Numeric answers in this notebook are produced by code that ran,
never quoted from memory; all randomness comes from `rng`.

In [1]:
import itertools

import numpy as np
from scipy.optimize import linprog

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

print("numpy", np.__version__)

numpy 2.4.6


## L0 — Concept Checks

### Problem L0.1: Why an LP Optimum Sits on the Boundary

**Problem Statement:** Prove that if $\mathbf{c}\neq\mathbf{0}$, the minimum of $\mathbf{c}^\top \mathbf{x}$
over a polyhedron $P$ can never be attained at an interior point of $P$. Then state the fundamental
theorem of linear programming.

*Intuition:* An affine function has a constant nonzero gradient, so from any interior point you can always step downhill and stay feasible.

**Solution:**

**Step 1 (the interior argument).** Let $\mathbf{x}_0 \in \operatorname{int}P$, so there is $\epsilon \gt 0$
with $B(\mathbf{x}_0,\epsilon)\subset P$. Take

$$
\mathbf{x}_1 = \mathbf{x}_0 - \frac{\epsilon}{2}\frac{\mathbf{c}}{\lVert \mathbf{c}\rVert} \in P
$$

Then

$$
\mathbf{c}^\top \mathbf{x}_1 = \mathbf{c}^\top \mathbf{x}_0 - \frac{\epsilon}{2}\lVert \mathbf{c}\rVert \lt \mathbf{c}^\top \mathbf{x}_0
$$

so $\mathbf{x}_0$ is not optimal. Hence any optimum lies on $\partial P$.

**Step 2 (iterate the argument).** Repeating on the boundary face reduces the dimension of the face
containing the candidate optimum by at least one each time (the objective restricted to a face is again
affine). After finitely many steps the argument terminates at a face on which the objective is constant,
whose own extreme points are extreme points of $P$.

**Step 3 (the fundamental theorem of LP).**

> If $P = \{\mathbf{x} : A\mathbf{x} = \mathbf{b},\ \mathbf{x}\ge\mathbf{0}\}$ is nonempty and
> $\min_{\mathbf{x}\in P}\mathbf{c}^\top \mathbf{x}$ is finite, then the minimum is attained at an extreme
> point (basic feasible solution) of $P$.

The standard-form hypothesis $\mathbf{x}\ge\mathbf{0}$ matters: it guarantees $P$ contains no line, hence
has extreme points. A polyhedron containing a line (e.g. all of $\mathbb{R}^n$) has none, and the theorem's
conclusion is vacuous there.

**Step 4 (algorithmic consequence).** The optimum can be found by searching a *finite* set — the at most
$\binom{n}{m}$ basic feasible solutions — which is exactly what the simplex method does, walking from
vertex to adjacent vertex along improving edges.

$$
\boxed{\mathbf{c}\neq\mathbf{0} \implies \text{no interior optimum}; \ \text{a finite optimum is attained at an extreme point}}
$$

> **Key takeaway:** Linearity turns a continuous optimization into a combinatorial one: the search space collapses from a polyhedron to its vertex set, which is the conceptual foundation of the simplex method.

### Problem L0.2: Convexity of a QP Is Exactly $Q \succeq 0$

**Problem Statement:** For $f(\mathbf{x}) = \frac12\mathbf{x}^\top Q\mathbf{x} + \mathbf{c}^\top \mathbf{x}$ with
$Q$ symmetric, show that $f$ is convex iff $Q \succeq 0$, and explain what goes wrong when $Q$ is
indefinite — even over a bounded polyhedron.

*Intuition:* A quadratic is its own second-order model, so the Hessian test is exact and global.

**Solution:**

**Step 1 (Hessian criterion).** $\nabla^2 f(\mathbf{x}) = Q$ for every $\mathbf{x}$. A twice-differentiable
function is convex iff its Hessian is PSD everywhere, so

$$
f \text{ convex} \iff Q \succeq 0
$$

**Step 2 (direct verification).** For $\mathbf{x},\mathbf{y}$ and $\theta\in[0,1]$, a short computation
gives the exact identity

$$
\theta f(\mathbf{x}) + (1-\theta)f(\mathbf{y}) - f\left(\theta\mathbf{x}+(1-\theta)\mathbf{y}\right) = \frac{\theta(1-\theta)}{2}\left(\mathbf{x}-\mathbf{y}\right)^\top Q\left(\mathbf{x}-\mathbf{y}\right)
$$

which is $\ge 0$ for all $\mathbf{x},\mathbf{y}$ precisely when $Q\succeq 0$. (Nonconvexity is exposed by
taking $\mathbf{x}-\mathbf{y}$ along a negative-curvature eigenvector.)

**Step 3 (what indefiniteness costs).** If $Q$ has a negative eigenvalue, minimizing $f$ over a
*polytope* is NP-hard in general. Two symptoms:

- Local minima proliferate on the boundary: with $Q = -I$ on the box $[-1,1]^n$ and $\mathbf{c}=\mathbf{0}$,
  $f(\mathbf{x}) = -\frac12\lVert \mathbf{x}\rVert^2$ has $2^n$ local minima, one at each vertex.
- Verifying that a candidate is globally optimal requires a global argument; KKT is only necessary, and
  the number of KKT points can be exponential.

Indeed, deciding whether an indefinite QP over a box has value below a threshold encodes MAX-CUT, hence the
hardness.

**Step 4 (the boundary case $Q\succeq0$ singular).** Convexity survives, but strict convexity and
uniqueness do not: $f$ is constant along $\ker Q \cap \{\mathbf{c}\perp\}$ and unbounded below along
$\ker Q$ if $\mathbf{c}$ has a component there ([optimization/02](../02_unconstrained_optimality_conditions/), Problem L3.4).

$$
\boxed{f \ \text{convex} \iff Q\succeq0; \ \ Q \ \text{indefinite} \implies \text{QP is NP-hard in general}}
$$

> **Key takeaway:** "Quadratic" is not the same as "tractable" — the entire tractability of QP rests on a single spectral condition, so checking $\lambda_{\min}(Q) \ge 0$ is the first step of any QP modeling exercise.

### Problem L0.3: The Convex Hierarchy in One Page

**Problem Statement:** Give explicit constructions establishing
$\text{LP} \subset \text{QP} \subset \text{SOCP} \subset \text{SDP}$, and state which class each of the
following belongs to: least squares, SVM, robust LP with ellipsoidal uncertainty, max-cut relaxation.

*Intuition:* Each step up the hierarchy widens the cone through which the feasible set is described: orthant, then ice-cream cone, then PSD cone.

**Solution:**

**Step 1 ($\text{LP}\subset\text{QP}$).** An LP is a QP with $Q = 0$:

$$
\min\ \mathbf{c}^\top \mathbf{x} \ \text{s.t.}\ A\mathbf{x}\le\mathbf{b} \ \longleftrightarrow\ \min\ \tfrac12\mathbf{x}^\top \mathbf{0}\mathbf{x} + \mathbf{c}^\top \mathbf{x} \ \text{s.t.}\ A\mathbf{x}\le\mathbf{b}
$$

and $0 \succeq 0$, so the QP is convex.

**Step 2 ($\text{QP}\subset\text{SOCP}$).** Write $Q = R^\top R$ (Cholesky or PSD square root) and use an
epigraph variable $t$:

$$
\min\ t + \mathbf{c}^\top \mathbf{x} \quad \text{s.t.}\quad \tfrac12\lVert R\mathbf{x}\rVert^2 \le t, \quad A\mathbf{x}\le\mathbf{b}
$$

The constraint $\lVert R\mathbf{x}\rVert^2 \le 2t$ is a **rotated second-order cone** constraint, and the
rotation identity converts it to a standard one:

$$
\lVert \mathbf{u}\rVert^2 \le yz,\ y,z\ge0 \iff \left\lVert \begin{pmatrix} 2\mathbf{u} \\ y - z\end{pmatrix}\right\rVert_2 \le y+z
$$

(take $\mathbf{u} = R\mathbf{x}$, $y = 2t$, $z = 1$).

**Step 3 ($\text{SOCP}\subset\text{SDP}$).** By the Schur complement (Problem L1.6),

$$
\lVert \mathbf{u}\rVert_2 \le t \iff \begin{bmatrix} t I & \mathbf{u} \\ \mathbf{u}^\top & t\end{bmatrix} \succeq 0
$$

an "arrow" LMI. Each SOC constraint becomes one linear matrix inequality, so any SOCP is an SDP.

**Step 4 (strictness).** Each inclusion is strict: $\min \lVert \mathbf{x}\rVert_2$ is SOCP but not QP-
representable with a *linear* objective and polyhedral constraints; and $\min \lambda_{\max}(F(\mathbf{x}))$
is SDP but not SOCP-representable in general.

**Step 5 (classification).**

| Problem | Class |
|---|---|
| Least squares, ridge regression | QP (unconstrained/convex) |
| Hard- and soft-margin SVM | QP (box-constrained dual) |
| Robust LP, ellipsoidal uncertainty | SOCP |
| Max-cut Goemans-Williamson relaxation | SDP |
| Chebyshev ($\ell_\infty$) approximation | LP |

$$
\boxed{\text{LP} \subset \text{QP} \subset \text{SOCP} \subset \text{SDP}, \ \text{via } Q=0,\ \text{epigraph} + \text{rotated cone},\ \text{Schur complement}}
$$

> **Key takeaway:** Modeling is the art of finding the *lowest* class that expresses your problem — solver cost and reliability degrade steadily as you climb, so recognizing an SOCP that is secretly a QP is worth real time.

### Problem L0.4: Simplex Is Fast in Practice and Exponential in Theory

**Problem Statement:** Explain the Klee-Minty construction and its consequence for simplex complexity, and
contrast with the polynomial guarantee of interior-point methods. How many vertices does a Klee-Minty cube
in $\mathbb{R}^n$ force simplex through, and what is that number for $n = 50$?

*Intuition:* A slightly tilted hypercube can be arranged so that the steepest-descent pivot rule visits every single vertex.

**Solution:**

**Step 1 (the construction).** The Klee-Minty cube is a perturbed unit cube in $\mathbb{R}^n$, e.g.

$$
\max\ \sum_{j=1}^n 10^{\,n-j}x_j \quad \text{s.t.}\quad 2\sum_{j=1}^{i-1}10^{\,i-j}x_j + x_i \le 100^{\,i-1}, \quad x_i \ge 0
$$

It is combinatorially a cube, so it has $2^n$ vertices; the perturbation orders them so that Dantzig's
most-negative-reduced-cost rule visits **all** of them.

**Step 2 (the count).** Simplex therefore performs $2^n - 1$ pivots in the worst case. For $n = 50$:

$$
2^{50} - 1 \approx 1.1\times 10^{15} \ \text{pivots}
$$

At $10^9$ pivots per second that is roughly $13$ days for a $50$-variable problem — worst case only, but a
genuine one.

**Step 3 (no rule is known to escape).** Analogous exponential families are known for most standard pivot
rules (Bland, steepest edge, greatest improvement). Whether *some* pivot rule is polynomial is open and is
tied to the polynomial Hirsch conjecture on the diameter of polytope graphs.

**Step 4 (interior-point contrast).** Primal-dual barrier methods follow the central path and reach
accuracy $\epsilon$ in

$$
O\!\left(\sqrt{n}\,\log\frac{1}{\epsilon}\right) \ \text{Newton iterations}
$$

each costing one linear solve. For $n = 50$ and $\epsilon = 10^{-8}$ that is a few dozen iterations,
independent of any vertex combinatorics.

**Step 5 (practice).** Simplex nevertheless remains competitive: on real instances it typically takes
$O(m)$ to $O(m\log n)$ pivots, and it warm-starts beautifully (essential in branch-and-bound). Smoothed
analysis (Spielman-Teng) proves that simplex is polynomial on *randomly perturbed* inputs, explaining the
gap between worst-case theory and observed behaviour.

$$
\boxed{\text{Klee-Minty: } 2^n - 1 \ \text{pivots} \ (1.1\times10^{15} \text{ for } n=50); \ \text{IPM: } O(\sqrt{n}\log(1/\epsilon))}
$$

> **Key takeaway:** "Fast in practice" and "polynomial time" are independent claims; LP is the textbook case where the practical algorithm is exponential in the worst case and the polynomial algorithm was invented decades later.

## L1 — Foundations

### Problem L1.1: Conversion to Standard Form

**Problem Statement:** Convert

$$
\min\ 2x_1 - 3x_2 + x_3 \quad \text{s.t.}\quad x_1 + x_2 \le 4,\quad 2x_1 - x_3 \ge 1,\quad x_1 + x_2 + x_3 = 3,\quad x_1 \ge 0,\ x_2 \le 0,\ x_3 \ \text{free}
$$

into standard form $\min\ \mathbf{c}^\top \mathbf{x}$ s.t. $A\mathbf{x}=\mathbf{b}$, $\mathbf{x}\ge\mathbf{0}$,
and count the resulting variables and constraints.

*Intuition:* Every LP can be pushed into a single canonical shape by three moves: slacks, surpluses, and variable splitting.

**Solution:**

**Step 1 (fix the variable signs).**

- $x_1 \ge 0$: keep as is.
- $x_2 \le 0$: substitute $x_2 = -x_2'$ with $x_2' \ge 0$.
- $x_3$ free: substitute $x_3 = x_3^+ - x_3^-$ with $x_3^\pm \ge 0$.

**Step 2 (fix the constraint types).**

- $x_1 + x_2 \le 4 \ \Rightarrow\ x_1 - x_2' + s_1 = 4$ with **slack** $s_1 \ge 0$.
- $2x_1 - x_3 \ge 1 \ \Rightarrow\ 2x_1 - x_3^+ + x_3^- - s_2 = 1$ with **surplus** $s_2 \ge 0$.
- The equality passes through: $x_1 - x_2' + x_3^+ - x_3^- = 3$.

**Step 3 (rewrite the objective).**

$$
2x_1 - 3x_2 + x_3 = 2x_1 + 3x_2' + x_3^+ - x_3^-
$$

**Step 4 (assemble).** With $\mathbf{x} = (x_1, x_2', x_3^+, x_3^-, s_1, s_2)^\top \ge \mathbf{0}$:

$$
\mathbf{c} = (2,\ 3,\ 1,\ -1,\ 0,\ 0)^\top , \qquad A = \begin{bmatrix} 1 & -1 & 0 & 0 & 1 & 0 \\ 2 & 0 & -1 & 1 & 0 & -1 \\ 1 & -1 & 1 & -1 & 0 & 0\end{bmatrix}, \qquad \mathbf{b} = \begin{pmatrix} 4 \\ 1 \\ 3\end{pmatrix}
$$

**Step 5 (count and remarks).** $n = 6$ variables, $m = 3$ equality constraints, all variables sign
constrained. The number of basic feasible solutions is at most $\binom{6}{3} = 20$. Note that a basic
solution never has both $x_3^+ \gt 0$ and $x_3^- \gt 0$ simultaneously (their columns are negatives of one
another, hence linearly dependent), so the splitting introduces no spurious solutions.

$$
\boxed{n = 6,\ m = 3; \ \text{slack } s_1, \text{ surplus } s_2, \text{ split } x_3 = x_3^+ - x_3^-, \ x_2 = -x_2'}
$$

> **Key takeaway:** Standard form costs at most one extra variable per inequality and per free variable, so the conversion is cheap — which is why every LP solver's internal representation is the standard form regardless of how you typed the model.

Verification: solve the original LP and the standard form of Step 4 independently and compare optimal values.

In [2]:
c_orig = np.array([2.0, -3.0, 1.0])
res_orig = linprog(c_orig,
                   A_ub=[[1.0, 1.0, 0.0], [-2.0, 0.0, 1.0]], b_ub=[4.0, -1.0],
                   A_eq=[[1.0, 1.0, 1.0]], b_eq=[3.0],
                   bounds=[(0, None), (None, 0), (None, None)], method="highs")

A_std = np.array([[1.0, -1.0, 0.0, 0.0, 1.0, 0.0],
                  [2.0, 0.0, -1.0, 1.0, 0.0, -1.0],
                  [1.0, -1.0, 1.0, -1.0, 0.0, 0.0]])
b_std = np.array([4.0, 1.0, 3.0])
c_std = np.array([2.0, 3.0, 1.0, -1.0, 0.0, 0.0])
res_std = linprog(c_std, A_eq=A_std, b_eq=b_std, bounds=[(0, None)] * 6, method="highs")

x1, x2p, x3p, x3m, s1, s2 = res_std.x
print(f"original form : x = {res_orig.x}, optimal value = {res_orig.fun:.10f}")
print(f"standard form : x = {res_std.x}, optimal value = {res_std.fun:.10f}")
print(f"recovered original variables: x1 = {x1:.4f}, x2 = {-x2p:.4f}, x3 = {x3p - x3m:.4f}")
print(f"n = {A_std.shape[1]} variables, m = {A_std.shape[0]} equalities, "
      f"at most C(6,3) = {len(list(itertools.combinations(range(6), 3)))} bases")
assert abs(res_orig.fun - res_std.fun) < 1e-9
assert np.allclose([x1, -x2p, x3p - x3m], res_orig.x, atol=1e-9)

original form : x = [1.3333 0.     1.6667], optimal value = 4.3333333333
standard form : x = [1.3333 0.     1.6667 0.     2.6667 0.    ], optimal value = 4.3333333333
recovered original variables: x1 = 1.3333, x2 = -0.0000, x3 = 1.6667
n = 6 variables, m = 3 equalities, at most C(6,3) = 20 bases


Both forms return the same optimal value to $10^{-9}$, and undoing the substitutions recovers the original
optimizer — confirming that slacks, surpluses, and splitting are value-preserving bijections.

### Problem L1.2: Writing and Checking an LP Dual

**Problem Statement:** Write the dual of

$$
\min\ 4x_1 + 3x_2 \quad \text{s.t.}\quad x_1 + x_2 \ge 3,\quad 2x_1 + x_2 \ge 4,\quad \mathbf{x}\ge\mathbf{0}
$$

Solve both, verify strong duality, and verify complementary slackness.

*Intuition:* Each primal constraint becomes a dual variable, each primal variable a dual constraint, and the sense of every inequality flips.

**Solution:**

**Step 1 (the dual).** For $\min\mathbf{c}^\top \mathbf{x}$ s.t. $A\mathbf{x}\ge\mathbf{b}$,
$\mathbf{x}\ge\mathbf{0}$, the dual is $\max\mathbf{b}^\top \mathbf{y}$ s.t. $A^\top \mathbf{y}\le\mathbf{c}$,
$\mathbf{y}\ge\mathbf{0}$:

$$
\max\ 3y_1 + 4y_2 \quad \text{s.t.}\quad y_1 + 2y_2 \le 4,\quad y_1 + y_2 \le 3,\quad \mathbf{y}\ge\mathbf{0}
$$

**Step 2 (solve the primal).** Candidate vertices of $\{x_1+x_2\ge3,\ 2x_1+x_2\ge4,\ \mathbf{x}\ge0\}$:

| Vertex | Feasible? | $4x_1+3x_2$ |
|---|---|---|
| $(0,4)$ | yes ($4\ge3$, $4\ge4$) | $12$ |
| $(1,2)$ (both tight) | yes | $10$ |
| $(3,0)$ | yes ($3\ge3$, $6\ge4$) | $12$ |

So $p^* = 10$ at $\mathbf{x}^* = (1,2)$.

**Step 3 (solve the dual).** Since $x_1^*, x_2^* \gt 0$, complementary slackness makes both dual
constraints tight:

$$
y_1 + 2y_2 = 4, \qquad y_1 + y_2 = 3 \ \Longrightarrow\ y_2 = 1,\ y_1 = 2
$$

Check dual feasibility: $\mathbf{y}^* = (2,1) \ge \mathbf{0}$ $\checkmark$. Objective
$3(2) + 4(1) = 10 = p^*$.

**Step 4 (strong duality).** $d^* = 10 = p^*$: both problems are feasible and bounded, so LP strong duality
holds automatically, no Slater condition required.

**Step 5 (all complementary slackness relations).**

- $y_1^* = 2 \gt 0 \Rightarrow x_1^*+x_2^* = 3$: $1+2 = 3$ $\checkmark$
- $y_2^* = 1 \gt 0 \Rightarrow 2x_1^*+x_2^* = 4$: $2+2 = 4$ $\checkmark$
- $x_1^* = 1 \gt 0 \Rightarrow y_1^*+2y_2^* = 4$: $2+2 = 4$ $\checkmark$
- $x_2^* = 2 \gt 0 \Rightarrow y_1^*+y_2^* = 3$: $2+1 = 3$ $\checkmark$

$$
\boxed{\mathbf{x}^* = (1,2),\ \mathbf{y}^* = (2,1),\ p^* = d^* = 10}
$$

> **Key takeaway:** The dual is not extra work — it is a *certificate generator*: exhibiting $\mathbf{y}^*$ with $\mathbf{b}^\top \mathbf{y}^* = \mathbf{c}^\top \mathbf{x}^*$ proves optimality without any search.

Verification: solve the primal and the dual independently with a library solver and check
$p^\star = d^\star = 10$ and all four complementary-slackness relations.

In [3]:
c_p = np.array([4.0, 3.0])
A_p = np.array([[1.0, 1.0], [2.0, 1.0]])
b_p = np.array([3.0, 4.0])

primal = linprog(c_p, A_ub=-A_p, b_ub=-b_p, bounds=[(0, None)] * 2, method="highs")
dual = linprog(-b_p, A_ub=A_p.T, b_ub=c_p, bounds=[(0, None)] * 2, method="highs")
x_s, y_s = primal.x, dual.x

print(f"primal x* = {x_s}, p* = {primal.fun:.6f}")
print(f"dual   y* = {y_s}, d* = {-dual.fun:.6f}")
print(f"duality gap                          = {abs(primal.fun + dual.fun):.2e}")
print(f"y_i * (A x - b)_i                    = {y_s * (A_p @ x_s - b_p)}")
print(f"x_j * (c - A^T y)_j                  = {x_s * (c_p - A_p.T @ y_s)}")
assert abs(primal.fun + dual.fun) < 1e-9
assert np.allclose(x_s, [1.0, 2.0], atol=1e-9) and np.allclose(y_s, [2.0, 1.0], atol=1e-9)
assert np.abs(y_s * (A_p @ x_s - b_p)).max() < 1e-9
assert np.abs(x_s * (c_p - A_p.T @ y_s)).max() < 1e-9

primal x* = [1. 2.], p* = 10.000000
dual   y* = [2. 1.], d* = 10.000000
duality gap                          = 0.00e+00
y_i * (A x - b)_i                    = [0. 0.]
x_j * (c - A^T y)_j                  = [0. 0.]


The solver reproduces $\mathbf{x}^* = (1,2)$ and $\mathbf{y}^* = (2,1)$ with $p^* = d^* = 10$, and every
complementary-slackness product is zero at machine precision, exactly as Steps 2-5 predicted by hand.

### Problem L1.3: Vertices Are Basic Feasible Solutions

**Problem Statement:** For $P = \{\mathbf{x}\in\mathbb{R}^2 : x_1 + x_2 \le 2,\ x_1 \le 1,\ \mathbf{x}\ge\mathbf{0}\}$,
enumerate all candidate vertices by choosing $2$ tight constraints at a time, discard the infeasible or
degenerate choices, and list the vertices. Explain the general counting bound.

*Intuition:* A vertex in $\mathbb{R}^n$ is pinned down by $n$ linearly independent active constraints — so enumerate subsets of size $n$.

**Solution:**

**Step 0 (the four constraints).**
$c_1: x_1+x_2 \le 2$; $c_2: x_1 \le 1$; $c_3: x_1 \ge 0$; $c_4: x_2 \ge 0$.

**Step 1 (enumerate all $\binom{4}{2}=6$ pairs).**

| Tight pair | Solution | Independent? | Feasible? | Verdict |
|---|---|---|---|---|
| $c_1, c_2$ | $(1,1)$ | yes | yes | **vertex** |
| $c_1, c_3$ | $(0,2)$ | yes | yes | **vertex** |
| $c_1, c_4$ | $(2,0)$ | yes | no ($x_1 = 2 \gt 1$) | reject |
| $c_2, c_3$ | inconsistent ($x_1 = 1$ and $x_1 = 0$) | dependent rows | — | reject |
| $c_2, c_4$ | $(1,0)$ | yes | yes | **vertex** |
| $c_3, c_4$ | $(0,0)$ | yes | yes | **vertex** |

**Step 2 (the vertex list).**

$$
V(P) = \{(0,0),\ (1,0),\ (1,1),\ (0,2)\}
$$

a quadrilateral. Its objective values can be tabulated to solve any LP over $P$ by inspection.

**Step 3 (the equivalence).** In standard form $\{A\mathbf{x}=\mathbf{b},\mathbf{x}\ge0\}$ with
$A\in\mathbb{R}^{m\times n}$ of rank $m$, a **basic feasible solution** picks $m$ linearly independent
columns (the basis $B$), sets the other $n-m$ variables to zero, and solves
$\mathbf{x}_B = A_B^{-1}\mathbf{b}$; it is feasible iff $\mathbf{x}_B \ge \mathbf{0}$. The theorem
(Problem L3.1) is that BFSs and extreme points coincide.

**Step 4 (counting).** At most

$$
\binom{n}{m} \ \text{bases}
$$

exist, so a polyhedron in standard form has finitely many vertices — here, in the inequality description,
at most $\binom{4}{2} = 6$, of which $4$ survive. The bound is loose because of infeasible and dependent
choices, and *degeneracy* (more than $n$ constraints active at one point) makes several bases describe the
same vertex.

$$
\boxed{V(P) = \{(0,0),(1,0),(1,1),(0,2)\}; \ \text{at most } \binom{n}{m} \ \text{basic solutions}}
$$

> **Key takeaway:** Vertex enumeration is exponential in general, which is why simplex *walks* the vertex graph instead of listing it — but for tiny problems the enumeration is a completely reliable hand method.

Verification: enumerate all $\binom{4}{2} = 6$ pairs of tight constraints, reject the dependent and the
infeasible ones, and compare the surviving list with Step 2.

In [4]:
G = np.array([[1.0, 1.0],    # c1: x1 + x2 <= 2
              [1.0, 0.0],    # c2: x1      <= 1
              [-1.0, 0.0],   # c3: -x1     <= 0
              [0.0, -1.0]])  # c4: -x2     <= 0
h = np.array([2.0, 1.0, 0.0, 0.0])

vertices = []
for i, j in itertools.combinations(range(4), 2):
    M = G[[i, j]]
    if abs(np.linalg.det(M)) < 1e-12:
        print(f"c{i+1}, c{j+1}: dependent rows -> reject")
        continue
    v = np.linalg.solve(M, h[[i, j]])
    feasible = bool((G @ v <= h + 1e-9).all())
    print(f"c{i+1}, c{j+1}: solution {v}  feasible = {feasible}")
    if feasible:
        vertices.append(v)

vertices = np.array(sorted(map(tuple, np.round(vertices, 12) + 0.0)))
print(f"\nvertex set V(P) = {vertices.tolist()}")
assert len(vertices) == 4
assert np.allclose(vertices, [[0.0, 0.0], [0.0, 2.0], [1.0, 0.0], [1.0, 1.0]], atol=1e-9)

c1, c2: solution [1. 1.]  feasible = True
c1, c3: solution [0. 2.]  feasible = True
c1, c4: solution [ 2. -0.]  feasible = False
c2, c3: dependent rows -> reject
c2, c4: solution [ 1. -0.]  feasible = True
c3, c4: solution [-0. -0.]  feasible = True

vertex set V(P) = [[0.0, 0.0], [0.0, 2.0], [1.0, 0.0], [1.0, 1.0]]


Six pairs, one dependent ($c_2, c_3$), one infeasible ($c_1, c_4$ gives $x_1 = 2 \gt 1$), four vertices —
$\{(0,0), (1,0), (1,1), (0,2)\}$, matching the hand enumeration.

### Problem L1.4: Least Squares and Ridge Regression as QPs

**Problem Statement:** Write ordinary least squares and ridge regression in the QP form
$\min \frac12\mathbf{x}^\top Q\mathbf{x} + \mathbf{c}^\top \mathbf{x}$, identify $Q$ and $\mathbf{c}$ in each case,
state when each is strictly convex, and write the nonnegative least squares variant as a constrained QP.

*Intuition:* Squared-error objectives are quadratics whose Hessian is the data Gram matrix, possibly shifted.

**Solution:**

**Step 1 (ordinary least squares).**

$$
\frac12\lVert X\mathbf{w}-\mathbf{y}\rVert^2 = \frac12\mathbf{w}^\top \left(X^\top X\right)\mathbf{w} - \left(X^\top \mathbf{y}\right)^\top \mathbf{w} + \frac12\lVert \mathbf{y}\rVert^2
$$

so $Q = X^\top X \succeq 0$ and $\mathbf{c} = -X^\top \mathbf{y}$ (the constant is irrelevant to the argmin). It is
an **unconstrained convex QP**, strictly convex iff $\operatorname{rank}(X) = n$; the optimality condition
is the normal equations $X^\top X\mathbf{w} = X^\top \mathbf{y}$.

**Step 2 (ridge regression).**

$$
\frac12\lVert X\mathbf{w}-\mathbf{y}\rVert^2 + \frac{\lambda}{2}\lVert \mathbf{w}\rVert^2 \ \Longrightarrow\ Q = X^\top X + \lambda I \succ 0 \ (\lambda \gt 0), \quad \mathbf{c} = -X^\top \mathbf{y}
$$

Strictly convex for every $\lambda \gt 0$ regardless of $\operatorname{rank}(X)$, with unique solution
$\mathbf{w}^* = (X^\top X+\lambda I)^{-1}X^\top \mathbf{y}$ and condition number
$\kappa = \frac{\sigma_1^2+\lambda}{\sigma_n^2+\lambda}$, which decreases in $\lambda$.

**Step 3 (nonnegative least squares).** Adding $\mathbf{w}\ge\mathbf{0}$ gives the constrained QP

$$
\min_{\mathbf{w}\ge\mathbf{0}}\ \tfrac12\mathbf{w}^\top \left(X^\top X\right)\mathbf{w} - \left(X^\top \mathbf{y}\right)^\top \mathbf{w}
$$

Now there is no closed form: the KKT system is
$X^\top X\mathbf{w} - X^\top \mathbf{y} = \boldsymbol{\mu} \ge \mathbf{0}$ with $\mu_iw_i = 0$, an LCP solved by
active-set methods (Lawson-Hanson) or projected gradient. The nonnegativity constraints induce **sparsity**
in $\mathbf{w}$ — the same active-set/sparsity link as the KKT complementarity of [optimization/06](../06_kkt_conditions_and_duality/).

**Step 4 (LASSO as a QP).** The $\ell_1$ problem
$\min\frac12\lVert X\mathbf{w}-\mathbf{y}\rVert^2 + \lambda\lVert \mathbf{w}\rVert_1$ becomes a QP after
splitting $\mathbf{w} = \mathbf{u}-\mathbf{v}$ with $\mathbf{u},\mathbf{v}\ge\mathbf{0}$:

$$
\min_{\mathbf{u},\mathbf{v}\ge\mathbf{0}}\ \tfrac12\lVert X(\mathbf{u}-\mathbf{v})-\mathbf{y}\rVert^2 + \lambda\mathbf{1}^\top (\mathbf{u}+\mathbf{v})
$$

doubling the variable count but staying inside the QP class.

$$
\boxed{\text{OLS}: Q = X^\top X; \quad \text{ridge}: Q = X^\top X+\lambda I \succ 0; \quad \text{NNLS}: \text{same } Q \ \text{with} \ \mathbf{w}\ge\mathbf{0}}
$$

> **Key takeaway:** Almost every linear-model estimator in statistics is one convex QP, differing only in the shift added to $Q$ and the polyhedron imposed on $\mathbf{w}$ — which is why a single QP solver covers the whole family.

Verification: build $Q$ and $\mathbf{c}$ for OLS, ridge, and NNLS on one deliberately near-collinear design,
solve each, and confirm the closed forms, the conditioning claim of Step 2, and the complementarity
conditions of Step 3.

In [5]:
from scipy.optimize import nnls

X = np.array([[1.0, 1.0], [1.0, 1.0001], [1.0, 1.0002], [1.0, 1.0003]])   # near-collinear on purpose
y_ls = np.array([1.0, 2.0, 2.0, -1.0])
lam = 0.5

Q_ols, c_ols = X.T @ X, -X.T @ y_ls
w_normal = np.linalg.solve(Q_ols, -c_ols)          # via the normal equations
w_lstsq = np.linalg.lstsq(X, y_ls, rcond=None)[0]  # via QR / SVD, no squaring
w_ridge = np.linalg.solve(X.T @ X + lam * np.eye(2), X.T @ y_ls)

sv = np.linalg.svd(X, compute_uv=False)
kappa_ols = (sv[0] / sv[-1]) ** 2
kappa_ridge = (sv[0] ** 2 + lam) / (sv[-1] ** 2 + lam)

print(f"singular values of X          = {sv}")
print(f"eigenvalues of Q = X^T X      = {np.linalg.eigvalsh(Q_ols)}")
print(f"w via normal equations        = {w_normal}")
print(f"w via lstsq (QR)              = {w_lstsq}")
print(f"  relative disagreement       = {np.abs(w_normal - w_lstsq).max() / np.abs(w_lstsq).max():.2e}")
print(f"  residual norms              = {np.linalg.norm(X @ w_normal - y_ls):.12f} vs "
      f"{np.linalg.norm(X @ w_lstsq - y_ls):.12f}")
print(f"w ridge (lambda = {lam})          = {w_ridge}")
print(f"cond(X^T X) = {kappa_ols:.3e}   cond(X^T X + lambda I) = {kappa_ridge:.3e}")

# NNLS: same Q, with w >= 0. Projected gradient against the library routine.
w_nnls = np.zeros(2)
step = 1.0 / np.linalg.eigvalsh(Q_ols)[-1]
for _ in range(200000):
    w_nnls = np.maximum(w_nnls - step * (Q_ols @ w_nnls + c_ols), 0.0)
grad = Q_ols @ w_nnls + c_ols
print(f"\nNNLS projected gradient       = {w_nnls}")
print(f"NNLS scipy.optimize.nnls      = {nnls(X, y_ls)[0]}")
print(f"KKT: gradient {np.round(grad, 8)} >= 0 and w_j * grad_j = {np.round(w_nnls * grad, 10)}")
assert np.abs(w_normal - w_lstsq).max() / np.abs(w_lstsq).max() < 1e-6
assert abs(np.linalg.norm(X @ w_normal - y_ls) - np.linalg.norm(X @ w_lstsq - y_ls)) < 1e-10
assert kappa_ridge < kappa_ols / 1e6
assert np.abs(w_nnls - nnls(X, y_ls)[0]).max() < 1e-6
assert (grad >= -1e-9).all() and np.abs(w_nnls * grad).max() < 1e-9

singular values of X          = [2.8286 0.0002]
eigenvalues of Q = X^T X      = [0.     8.0012]
w via normal equations        = [ 6001.8999 -5999.9999]
w via lstsq (QR)              = [ 6001.9 -6000. ]
  relative disagreement       = 1.14e-08
  residual norms              = 2.049390153192 vs 2.049390153192
w ridge (lambda = 0.5)          = [0.4708 0.4703]
cond(X^T X) = 3.201e+08   cond(X^T X + lambda I) = 1.700e+01



NNLS projected gradient       = [1. 0.]
NNLS scipy.optimize.nnls      = [1. 0.]
KKT: gradient [-0.      0.0003] >= 0 and w_j * grad_j = [-0.  0.]


The normal equations and the QR-based `lstsq` agree only to a relative $10^{-8}$ here — the visible price of
$\kappa(X^\top X) = \kappa(X)^2 \approx 3 \times 10^{8}$, which is why Step 2's conditioning remark matters —
even though both produce the same residual norm. The ridge shift drops the condition number by seven orders
of magnitude and moves the solution from $\mathbf{w} \approx (6002, -6000)$ to a modest vector, and the
projected-gradient NNLS solution matches `scipy.optimize.nnls` while satisfying the complementarity
conditions $w_j\,(Qw+c)_j = 0$ of Step 3.

### Problem L1.5: The Second-Order Cone and the Rotated Cone

**Problem Statement:** Prove that $\mathcal{Q}^{n+1} = \{(\mathbf{u},t)\in\mathbb{R}^n\times\mathbb{R} : \lVert \mathbf{u}\rVert_2 \le t\}$
is a closed convex cone. Then prove the rotated-cone identity

$$
\lVert \mathbf{u}\rVert_2^2 \le yz,\ y\ge0,\ z\ge0 \iff \left\lVert \begin{pmatrix} 2\mathbf{u}\\ y-z\end{pmatrix}\right\rVert_2 \le y+z
$$

*Intuition:* The cone is the epigraph of a norm, and the rotation is nothing but the algebraic identity $(y+z)^2 - (y-z)^2 = 4yz$.

**Solution:**

**Step 1 (cone property).** If $(\mathbf{u},t)\in\mathcal{Q}$ and $\theta \ge 0$, then
$\lVert \theta\mathbf{u}\rVert = \theta\lVert \mathbf{u}\rVert \le \theta t$, so
$\theta(\mathbf{u},t)\in\mathcal{Q}$: the set is closed under nonnegative scaling.

**Step 2 (convexity).** Let $(\mathbf{u}_1,t_1),(\mathbf{u}_2,t_2)\in\mathcal{Q}$ and $\theta\in[0,1]$. By
the triangle inequality and absolute homogeneity of the norm,

$$
\left\lVert \theta\mathbf{u}_1+(1-\theta)\mathbf{u}_2\right\rVert \le \theta\lVert \mathbf{u}_1\rVert + (1-\theta)\lVert \mathbf{u}_2\rVert \le \theta t_1 + (1-\theta)t_2
$$

so the convex combination is in $\mathcal{Q}$. (Any norm works; the Euclidean case is the "ice-cream"
cone.)

**Step 3 (closedness).** $\mathcal{Q} = \{(\mathbf{u},t) : \lVert \mathbf{u}\rVert - t \le 0\}$ is a
sublevel set of the continuous function $(\mathbf{u},t)\mapsto\lVert \mathbf{u}\rVert - t$, hence closed.

**Step 4 (the rotated identity).** Square both sides of the right-hand condition — legitimate because
$y+z \ge 0$ is implied whenever the norm inequality holds:

$$
4\lVert \mathbf{u}\rVert^2 + (y-z)^2 \le (y+z)^2
$$

Using $(y+z)^2 - (y-z)^2 = 4yz$, this is exactly

$$
4\lVert \mathbf{u}\rVert^2 \le 4yz \iff \lVert \mathbf{u}\rVert^2 \le yz
$$

For the sign conditions: the norm inequality forces $y + z \ge \lvert y-z\rvert$, which is equivalent to
$y \ge 0$ and $z \ge 0$. Both directions therefore hold.

**Step 5 (why it matters).** Any convex quadratic constraint $\lVert R\mathbf{x}\rVert^2 \le 2t$ is a
rotated cone constraint with $\mathbf{u}=R\mathbf{x}$, $y=2t$, $z=1$, hence SOC-representable — this is the
mechanism of the embedding $\text{QP}\subset\text{SOCP}$ (Problem L0.3). Likewise
$\lVert A\mathbf{x}+\mathbf{b}\rVert_2 \le \mathbf{c}^\top \mathbf{x}+d$ is directly a (composed with affine
map) SOC constraint, and affine preimages of convex cones are convex.

$$
\boxed{\mathcal{Q}^{n+1} \ \text{is a closed convex cone}; \quad \lVert \mathbf{u}\rVert^2\le yz,\ y,z\ge0 \iff \left\lVert (2\mathbf{u},\, y-z)\right\rVert_2 \le y+z}
$$

> **Key takeaway:** All the expressive power of SOCP comes from one shape — the epigraph of the Euclidean norm — plus the fact that affine substitution preserves convexity and conic structure.

Verification: sample random $(\mathbf{u}, y, z)$ from `rng` and check that the two sides of the
rotated-cone identity agree as *sets* — the left condition holds exactly when the right one does.

In [6]:
trials = 200000
u_s = rng.normal(size=(trials, 3))
y_s = rng.normal(size=trials) * 2.0
z_s = rng.normal(size=trials) * 2.0

left = (np.einsum("ij,ij->i", u_s, u_s) <= y_s * z_s) & (y_s >= 0) & (z_s >= 0)
stacked = np.column_stack([2.0 * u_s, y_s - z_s])
right = np.linalg.norm(stacked, axis=1) <= y_s + z_s + 1e-12

print(f"samples                              : {trials}")
print(f"left condition true                  : {left.sum()}")
print(f"right condition true                 : {right.sum()}")
print(f"disagreements                        : {(left != right).sum()}")
print(f"algebraic residual max |(y+z)^2-(y-z)^2-4yz| = "
      f"{np.abs((y_s + z_s)**2 - (y_s - z_s)**2 - 4 * y_s * z_s).max():.3e}")
assert (left == right).all()

samples                              : 200000
left condition true                  : 19435
right condition true                 : 19435
disagreements                        : 0
algebraic residual max |(y+z)^2-(y-z)^2-4yz| = 3.553e-14


Out of $200\,000$ random triples the two conditions never disagree, and the identity
$(y+z)^2 - (y-z)^2 = 4yz$ holds at machine-epsilon level — so the rotated cone really is the second-order
cone in rotated coordinates.

### Problem L1.6: The Schur Complement Lemma

**Problem Statement:** Prove: for symmetric $M = \begin{bmatrix} A & B \\ B^\top & C\end{bmatrix}$ with
$A \succ 0$,

$$
M \succeq 0 \iff C - B^\top A^{-1}B \succeq 0
$$

Then use it to express $\lVert \mathbf{u}\rVert_2 \le t$ and the quadratic constraint
$\mathbf{x}^\top P\mathbf{x} \le t$ ($P\succ0$) as linear matrix inequalities.

*Intuition:* Block-diagonalize by completing the square in block form; the leftover block is the Schur complement.

**Solution:**

**Step 1 (block completion of the square).** With $A\succ0$ define the invertible
$T = \begin{bmatrix} I & -A^{-1}B \\ 0 & I\end{bmatrix}$. Then a direct computation gives the congruence

$$
T^\top MT = \begin{bmatrix} A & 0 \\ 0 & C - B^\top A^{-1}B\end{bmatrix}
$$

**Step 2 (congruence preserves the sign).** For invertible $T$, $M \succeq 0 \iff T^\top MT\succeq0$ (write
$\mathbf{z} = T\mathbf{w}$; as $\mathbf{w}$ ranges over $\mathbb{R}^{n+m}$ so does $\mathbf{z}$). A block
diagonal matrix is PSD iff each block is, and $A\succ0$ by hypothesis. Hence

$$
M\succeq0 \iff C - B^\top A^{-1}B \succeq 0
$$

**Step 3 (equivalent quadratic-form proof).** Explicitly, for $\mathbf{z} = (\mathbf{p},\mathbf{q})$,

$$
\mathbf{z}^\top M\mathbf{z} = \left(\mathbf{p}+A^{-1}B\mathbf{q}\right)^\top A\left(\mathbf{p}+A^{-1}B\mathbf{q}\right) + \mathbf{q}^\top \left(C - B^\top A^{-1}B\right)\mathbf{q}
$$

Minimizing over $\mathbf{p}$ kills the first term, leaving the Schur complement form.

**Step 4 (SOC as an LMI).** For $t \gt 0$, apply the lemma to
$M = \begin{bmatrix} tI & \mathbf{u} \\ \mathbf{u}^\top & t\end{bmatrix}$ with $A = tI \succ 0$:

$$
M\succeq0 \iff t - \mathbf{u}^\top \left(\frac{1}{t}I\right)\mathbf{u} \ge 0 \iff t^2 \ge \lVert \mathbf{u}\rVert^2 \iff \lVert \mathbf{u}\rVert_2 \le t
$$

(The case $t = 0$ is handled by closedness / continuity.) This is the arrow-matrix embedding
$\text{SOCP}\subset\text{SDP}$.

**Step 5 (quadratic constraint as an LMI).** For $P \succ 0$, take
$M = \begin{bmatrix} P^{-1} & \mathbf{x} \\ \mathbf{x}^\top & t\end{bmatrix}$:

$$
M\succeq0 \iff t - \mathbf{x}^\top P\mathbf{x} \ge 0 \iff \mathbf{x}^\top P\mathbf{x}\le t
$$

so any convex quadratic constraint, and hence any convex QP, is an SDP — and the same trick expresses
matrix-fractional and quadratic-over-linear functions in LMI form.

$$
\boxed{\begin{bmatrix} A & B\\ B^\top & C\end{bmatrix}\succeq0 \ (A\succ0) \iff C - B^\top A^{-1}B\succeq0; \quad \lVert \mathbf{u}\rVert\le t \iff \begin{bmatrix} tI & \mathbf{u}\\ \mathbf{u}^\top & t\end{bmatrix}\succeq0}
$$

> **Key takeaway:** The Schur complement is the universal translator into LMI form — it is how norms, quadratics, and ratios all become semidefinite constraints, and therefore how one SDP solver covers the entire hierarchy.

Verification: check the Schur-complement equivalence on random symmetric blocks, then confirm the two LMI
rewrites of Steps 4 and 5.

In [7]:
agree = 0
for _ in range(2000):
    Ab = rng.normal(size=(2, 2))
    Ab = Ab @ Ab.T + 0.05 * np.eye(2)            # A > 0
    Bb = rng.normal(size=(2, 2))
    Cb = rng.normal(size=(2, 2))
    Cb = (Cb + Cb.T) / 2
    Mb = np.block([[Ab, Bb], [Bb.T, Cb]])
    lhs = np.linalg.eigvalsh(Mb)[0] >= -1e-10
    rhs = np.linalg.eigvalsh(Cb - Bb.T @ np.linalg.inv(Ab) @ Bb)[0] >= -1e-10
    agree += int(lhs == rhs)
print(f"Schur equivalence held on {agree}/2000 random blocks with A > 0")
assert agree == 2000

u_v, t_v = np.array([0.6, 0.8, 0.0]), 1.0        # ||u|| = 1 = t, boundary case
M_arrow = np.block([[t_v * np.eye(3), u_v.reshape(-1, 1)],
                    [u_v.reshape(1, -1), np.array([[t_v]])]])
P_m = np.array([[2.0, 0.5], [0.5, 1.0]])
x_v = np.array([0.3, -0.4])
t_q = float(x_v @ P_m @ x_v)
M_quad = np.block([[np.linalg.inv(P_m), x_v.reshape(-1, 1)],
                   [x_v.reshape(1, -1), np.array([[t_q]])]])
print(f"||u|| = {np.linalg.norm(u_v):.4f}, t = {t_v}; arrow-matrix eigenvalues = "
      f"{np.linalg.eigvalsh(M_arrow)}")
print(f"x^T P x = {t_q:.6f}; quadratic LMI eigenvalues = {np.linalg.eigvalsh(M_quad)}")
assert np.linalg.eigvalsh(M_arrow)[0] > -1e-10 and np.linalg.eigvalsh(M_quad)[0] > -1e-10

Schur equivalence held on 2000/2000 random blocks with A > 0
||u|| = 1.0000, t = 1.0; arrow-matrix eigenvalues = [0. 1. 1. 2.]
x^T P x = 0.220000; quadratic LMI eigenvalues = [-0.      0.4805  1.4538]


The equivalence holds on all $2000$ random blocks with $A \succ 0$; the arrow matrix built from a boundary
point $\lVert \mathbf{u}\rVert = t$ is PSD with a zero eigenvalue, and the matrix of Step 5 is PSD exactly
when $t = \mathbf{x}^\top P\mathbf{x}$ — both LMI rewrites confirmed.

## L2 — Applications (AI/ML and Physics)

### Problem L2.1: Production Planning and Shadow Prices

**Problem Statement:** A shop makes two products with profits $5$ and $4$ per unit. Product 1 uses $6$
machine-hours and $1$ labour-hour; product 2 uses $4$ machine-hours and $2$ labour-hours. There are $24$
machine-hours and $6$ labour-hours available. Solve the LP, solve the dual, and interpret the dual
variables. What would you pay for one extra machine-hour?

*Intuition:* The dual prices each resource so that no product is profitable to make at those prices, and the cheapest such pricing equals the maximum profit.

**Solution:**

**Step 1 (the model).**

$$
\max\ 5x_1 + 4x_2 \quad \text{s.t.}\quad 6x_1 + 4x_2 \le 24,\quad x_1 + 2x_2 \le 6,\quad \mathbf{x}\ge\mathbf{0}
$$

**Step 2 (vertex enumeration).**

| Vertex | Feasible? | Profit |
|---|---|---|
| $(0,0)$ | yes | $0$ |
| $(4,0)$ | yes ($24\le24$, $4\le6$) | $20$ |
| $(3, 1.5)$ (both tight) | yes | $21$ |
| $(0,3)$ | yes ($12\le24$, $6\le6$) | $12$ |

So $p^* = 21$ at $\mathbf{x}^* = (3, 1.5)$, using **all** machine-hours and **all** labour-hours.

**Step 3 (the dual).**

$$
\min\ 24y_1 + 6y_2 \quad \text{s.t.}\quad 6y_1 + y_2 \ge 5,\quad 4y_1 + 2y_2 \ge 4,\quad \mathbf{y}\ge\mathbf{0}
$$

Since $x_1^*,x_2^* \gt 0$, complementary slackness makes both dual constraints tight:

$$
6y_1 + y_2 = 5, \qquad 2y_1 + y_2 = 2 \ \Longrightarrow\ 4y_1 = 3,\ y_1 = 0.75,\ y_2 = 0.5
$$

Dual objective: $24(0.75) + 6(0.5) = 18 + 3 = 21 = p^*$ $\checkmark$.

**Step 4 (interpretation).** $y_1^* = 0.75$ is the **shadow price of a machine-hour** and $y_2^* = 0.5$ that
of a labour-hour. Both are positive because both resources are fully consumed (binding constraints).

**Step 5 (paying for capacity).** One extra machine-hour ($b_1 = 25$) raises the optimum by $y_1^* = 0.75$,
so you should pay anything **less than $0.75$** per machine-hour. Verify directly: with $b_1 = 25$ the
tight-tight vertex moves to $6x_1+4x_2=25$, $x_1+2x_2=6$, giving $x_1 = 3.25$, $x_2 = 1.375$ and profit
$16.25 + 5.5 = 21.75 = 21 + 0.75$ $\checkmark$. The shadow price is valid only over a *range* of $b_1$ — it
changes when the optimal basis changes (here, when $x_2$ would be driven to $0$ at $b_1 = 36$).

$$
\boxed{\mathbf{x}^*=(3,1.5),\ p^* = 21,\ \mathbf{y}^*=(0.75,\ 0.5); \ \text{one extra machine-hour is worth } 0.75}
$$

> **Key takeaway:** LP duality turns a planning model into a pricing model; the shadow prices tell you where to invest, and their validity range tells you how far that advice extends.

Verification: solve the production LP and its dual, then re-solve with $b_1 = 25$ to confirm the shadow
price $y_1^\star = 0.75$ predicts the change in the optimum.

In [8]:
c_pp = np.array([-5.0, -4.0])                    # maximize 5x1 + 4x2
A_pp = np.array([[6.0, 4.0], [1.0, 2.0]])
b_pp = np.array([24.0, 6.0])

prod = linprog(c_pp, A_ub=A_pp, b_ub=b_pp, bounds=[(0, None)] * 2, method="highs")
dual_pp = linprog(b_pp, A_ub=-A_pp.T, b_ub=c_pp, bounds=[(0, None)] * 2, method="highs")
prod25 = linprog(c_pp, A_ub=A_pp, b_ub=np.array([25.0, 6.0]), bounds=[(0, None)] * 2, method="highs")

print(f"x*  = {prod.x},  profit = {-prod.fun:.6f}")
print(f"y*  = {dual_pp.x},  dual objective = {dual_pp.fun:.6f}")
print(f"slack of the two resources = {b_pp - A_pp @ prod.x} (both zero: binding)")
print(f"b1 = 25: x* = {prod25.x}, profit = {-prod25.fun:.6f}")
print(f"profit increase = {-prod25.fun + prod.fun:.6f}   shadow price y1* = {dual_pp.x[0]:.6f}")
assert abs(prod.fun + 21.0) < 1e-9 and np.allclose(dual_pp.x, [0.75, 0.5], atol=1e-9)
assert abs((-prod25.fun + prod.fun) - dual_pp.x[0]) < 1e-9

x*  = [3.  1.5],  profit = 21.000000
y*  = [0.75 0.5 ],  dual objective = 21.000000
slack of the two resources = [0. 0.] (both zero: binding)
b1 = 25: x* = [3.25  1.375], profit = 21.750000
profit increase = 0.750000   shadow price y1* = 0.750000


The optimum is $\mathbf{x}^* = (3, 1.5)$ with profit $21$, the dual gives $\mathbf{y}^* = (0.75, 0.5)$ with
the same value, and adding one machine-hour raises the profit by exactly $0.75$ — the shadow price, verified
rather than asserted.

### Problem L2.2: The Diet Problem and Its Dual Pill-Seller

**Problem Statement:** Minimize the cost $2x_1 + 3x_2$ of two foods subject to nutrient requirements
$x_1 + 3x_2 \ge 6$ (protein) and $2x_1 + x_2 \ge 4$ (iron), $\mathbf{x}\ge\mathbf{0}$. Solve the primal and
the dual, and give the classic economic interpretation of the dual.

*Intuition:* The dual is a competitor selling synthetic nutrients who must price them so that no food is undercut, while maximizing the bill for your requirements.

**Solution:**

**Step 1 (primal solution).** Vertices of the feasible region:

| Vertex | Feasible? | Cost |
|---|---|---|
| $(6,0)$ | yes ($6\ge6$, $12\ge4$) | $12$ |
| $(1.2,\ 1.6)$ (both tight) | yes | $2.4 + 4.8 = 7.2$ |
| $(0,4)$ | yes ($12\ge6$, $4\ge4$) | $12$ |

So $p^* = 7.2$ at $\mathbf{x}^* = (1.2, 1.6)$.

**Step 2 (the dual).**

$$
\max\ 6y_1 + 4y_2 \quad \text{s.t.}\quad y_1 + 2y_2 \le 2,\quad 3y_1 + y_2 \le 3,\quad \mathbf{y}\ge\mathbf{0}
$$

Since both foods are purchased ($x_1^*, x_2^* \gt 0$), both dual constraints bind:

$$
y_1 + 2y_2 = 2, \qquad 3y_1 + y_2 = 3 \ \Longrightarrow\ y_1 = 0.8,\ y_2 = 0.6
$$

with objective $6(0.8) + 4(0.6) = 4.8 + 2.4 = 7.2 = p^*$ $\checkmark$.

**Step 3 (the pill-seller story).** Imagine a vendor selling pure protein at price $y_1$ per unit and pure
iron at $y_2$ per unit. To be competitive, the pills replicating one unit of food $j$ must not cost more
than food $j$ itself:

$$
\underbrace{y_1 + 2y_2}_{\text{pills matching food 1}} \le \underbrace{2}_{\text{price of food 1}}, \qquad 3y_1 + y_2 \le 3
$$

Subject to that, the vendor maximizes revenue $6y_1 + 4y_2$ from your requirement of $6$ protein and $4$
iron. Strong duality says the vendor can extract *exactly* your minimal food cost — no more, no less.

**Step 4 (marginal reading).** $y_1^* = 0.8$ means one extra unit of required protein raises your minimal
cost by $0.8$; $y_2^* = 0.6$ likewise for iron. These are the *marginal* nutrient values implied by the
food prices and the optimal mix.

**Step 5 (why some foods go unused).** A food $j$ with $\sum_i a_{ij}y_i^* \lt c_j$ (strictly cheaper to
replicate with pills than to buy) has $x_j^* = 0$ by complementary slackness — the LP's way of saying "this
food is dominated by a combination of the others".

$$
\boxed{\mathbf{x}^* = (1.2, 1.6),\ p^* = 7.2; \ \mathbf{y}^* = (0.8,\ 0.6) \ \text{are the implicit nutrient prices}}
$$

> **Key takeaway:** Dantzig's diet problem is the historical origin of LP, and its dual is the reason economists adopted linear programming: the multipliers are literally market-clearing prices.

Verification: solve the diet LP and its dual and confirm the nutrient prices of Steps 2 and 4.

In [9]:
c_d = np.array([2.0, 3.0])
A_d = np.array([[1.0, 3.0], [2.0, 1.0]])
b_d = np.array([6.0, 4.0])

diet = linprog(c_d, A_ub=-A_d, b_ub=-b_d, bounds=[(0, None)] * 2, method="highs")
pills = linprog(-b_d, A_ub=A_d.T, b_ub=c_d, bounds=[(0, None)] * 2, method="highs")

print(f"foods bought x* = {diet.x},  minimum cost p* = {diet.fun:.6f}")
print(f"pill prices  y* = {pills.x},  vendor revenue d* = {-pills.fun:.6f}")
print(f"nutrient surpluses A x* - b = {A_d @ diet.x - b_d} (both zero: both requirements bind)")
print(f"food-level slack c - A^T y* = {c_d - A_d.T @ pills.x} (zero where x_j* > 0)")
assert abs(diet.fun - 7.2) < 1e-9 and np.allclose(pills.x, [0.8, 0.6], atol=1e-9)
assert abs(diet.fun + pills.fun) < 1e-9

foods bought x* = [1.2 1.6],  minimum cost p* = 7.200000
pill prices  y* = [0.8 0.6],  vendor revenue d* = 7.200000
nutrient surpluses A x* - b = [0. 0.] (both zero: both requirements bind)
food-level slack c - A^T y* = [0. 0.] (zero where x_j* > 0)


The minimal diet costs $7.2$ at $\mathbf{x}^* = (1.2, 1.6)$, and the pill-seller's optimal prices
$\mathbf{y}^* = (0.8, 0.6)$ extract exactly that amount — strong duality and complementary slackness both
confirmed numerically.

### Problem L2.3: Markowitz Mean-Variance Portfolio as a QP

**Problem Statement:** Solve $\min \frac12\mathbf{w}^\top \Sigma\mathbf{w}$ subject to $\mathbf{1}^\top \mathbf{w}=1$
and $\boldsymbol{\mu}^\top \mathbf{w} = r$ (target return), with $\Sigma\succ0$. Derive the closed form, the
efficient frontier $\sigma^2(r)$, and identify the minimum-variance portfolio.

*Intuition:* Two linear constraints give two multipliers; solving the resulting $2\times2$ system produces the classical hyperbola of the efficient frontier.

**Solution:**

**Step 1 (KKT).** Constraints enter the Lagrangian with a **plus**, following
[`docs/notation.md`](../../docs/notation.md):

$$
\mathcal{L}(\mathbf{w},\lambda,\nu) = \tfrac12\mathbf{w}^\top \Sigma\mathbf{w} + \lambda\left(\boldsymbol{\mu}^\top \mathbf{w}-r\right) + \nu\left(\mathbf{1}^\top \mathbf{w}-1\right)
$$

$$
\nabla_{\mathbf{w}}\mathcal{L} = \Sigma\mathbf{w} + \lambda\boldsymbol{\mu} + \nu\mathbf{1} = \mathbf{0}
\ \Longrightarrow\ \mathbf{w} = \alpha\,\Sigma^{-1}\boldsymbol{\mu} + \beta\,\Sigma^{-1}\mathbf{1},
\qquad \alpha := -\lambda,\ \ \beta := -\nu
$$

The optimal portfolio is a combination of just **two** fixed vectors — the two-fund separation theorem.

**Step 2 (impose the constraints).** Define the standard scalars

$$
A = \mathbf{1}^\top \Sigma^{-1}\mathbf{1}, \qquad B = \mathbf{1}^\top \Sigma^{-1}\boldsymbol{\mu}, \qquad C = \boldsymbol{\mu}^\top \Sigma^{-1}\boldsymbol{\mu}, \qquad D = AC - B^2
$$

(One shows $D \gt 0$ unless $\boldsymbol{\mu}$ is proportional to $\mathbf{1}$, by Cauchy-Schwarz in the
$\Sigma^{-1}$ inner product.) Then

$$
\boldsymbol{\mu}^\top \mathbf{w} = \alpha C + \beta B = r, \qquad \mathbf{1}^\top \mathbf{w} = \alpha B + \beta A = 1
$$

Solving the $2\times2$ system:

$$
\alpha = \frac{Ar - B}{D}, \qquad \beta = \frac{C - Br}{D}
$$

**Step 3 (the efficient frontier).** Since $\Sigma\mathbf{w} = \alpha\boldsymbol{\mu} + \beta\mathbf{1}$,
the optimal variance is
$\sigma^2 = \mathbf{w}^\top \Sigma\mathbf{w} = \alpha\,\boldsymbol{\mu}^\top \mathbf{w} + \beta\,\mathbf{1}^\top \mathbf{w} = \alpha r + \beta$,
giving

$$
\sigma^2(r) = \frac{Ar^2 - 2Br + C}{D}
$$

a **parabola in $(r,\sigma^2)$**, equivalently a hyperbola in the $(\sigma, r)$ plane — the classical
Markowitz bullet.

**Step 4 (the global minimum-variance portfolio).** Minimize over $r$:

$$
\frac{d\sigma^2}{dr} = \frac{2Ar - 2B}{D} = 0 \ \Longrightarrow\ r_{\mathrm{mv}} = \frac{B}{A}, \qquad \sigma^2_{\mathrm{mv}} = \frac{A(B/A)^2 - 2B^2/A + C}{D} = \frac{1}{A}
$$

and at that point $\alpha = 0$, so $\mathbf{w}_{\mathrm{mv}} = \Sigma^{-1}\mathbf{1}/A$ — the return
constraint has zero shadow price because it is not restricting anything.

**Step 5 (reading the multipliers).** The *objective* of this program is the half-variance
$p^\star(r) = \tfrac12\sigma^2(r)$, so the sensitivity theorem $dp^\star/dr = -\lambda^\star = \alpha$ says

$$
\alpha = \frac{Ar-B}{D} = \frac{1}{2}\frac{d\sigma^2}{dr}, \qquad \text{equivalently}\qquad \frac{d\sigma^2}{dr} = 2\alpha .
$$

So $\alpha$ is the marginal **half-variance** cost of one more unit of expected return, and $2\alpha$ is the
marginal variance cost: zero at $r_{\mathrm{mv}}$, positive above it, negative below. Adding
$\mathbf{w}\ge\mathbf{0}$ (no short selling) destroys the closed form and turns the problem into a genuinely
constrained QP requiring a numerical solve.

$$
\boxed{\mathbf{w}^* = \alpha\Sigma^{-1}\boldsymbol{\mu} + \beta\Sigma^{-1}\mathbf{1}, \quad \sigma^2(r) = \frac{Ar^2-2Br+C}{D}, \quad \sigma^2_{\mathrm{mv}} = \frac1A, \quad \frac{d\sigma^2}{dr} = 2\alpha}
$$

> **Key takeaway:** The efficient frontier is not an empirical curve but the *value function* of a QP, and two-fund separation is just the statement that the KKT solution is affine in the multipliers.

Verification on a three-asset instance: solve the QP by its KKT system, compare with the closed form of
Steps 2-3, and check the sensitivity identity $dp^\star/dr = \alpha$ by finite differences.

In [10]:
Sigma = np.array([[0.10, 0.02, 0.01],
                  [0.02, 0.08, 0.03],
                  [0.01, 0.03, 0.12]])
mu = np.array([0.07, 0.10, 0.15])
ones = np.ones(3)
Si = np.linalg.inv(Sigma)
Acf, Bcf, Ccf = ones @ Si @ ones, ones @ Si @ mu, mu @ Si @ mu
Dcf = Acf * Ccf - Bcf**2


def frontier(r):
    """Closed form of Steps 1-3."""
    alpha = (Acf * r - Bcf) / Dcf
    beta = (Ccf - Bcf * r) / Dcf
    w = alpha * (Si @ mu) + beta * (Si @ ones)
    return w, alpha, beta


def kkt_solve(r):
    """Direct solve of the equality-constrained QP via its KKT matrix."""
    G = np.vstack([mu, ones])
    K = np.block([[Sigma, G.T], [G, np.zeros((2, 2))]])
    rhs = np.concatenate([np.zeros(3), [r, 1.0]])
    sol = np.linalg.solve(K, rhs)
    return sol[:3], sol[3:]


r0 = 0.11
w_cf, alpha, beta = frontier(r0)
w_kkt, lam = kkt_solve(r0)
print(f"A = {Acf:.4f}, B = {Bcf:.4f}, C = {Ccf:.4f}, D = {Dcf:.4f}")
print(f"closed-form w* = {w_cf}")
print(f"KKT solve   w* = {w_kkt}   (max diff {np.abs(w_cf - w_kkt).max():.2e})")
print(f"alpha = {alpha:.6f},  -lambda from KKT = {-lam[0]:.6f}")

var_cf = (Acf * r0**2 - 2 * Bcf * r0 + Ccf) / Dcf
h = 1e-6
dp = (0.5 * (Acf * (r0 + h)**2 - 2 * Bcf * (r0 + h) + Ccf) / Dcf
      - 0.5 * (Acf * (r0 - h)**2 - 2 * Bcf * (r0 - h) + Ccf) / Dcf) / (2 * h)
print(f"sigma^2(r0) closed form = {var_cf:.8f},  w*^T Sigma w* = {w_cf @ Sigma @ w_cf:.8f}")
print(f"dp*/dr by finite difference = {dp:.8f}   alpha = {alpha:.8f}   (must agree)")
print(f"d sigma^2/dr = {2 * alpha:.8f} = 2 * alpha")

r_mv = Bcf / Acf
w_mv, alpha_mv, _ = frontier(r_mv)
print(f"\nminimum-variance return r_mv = B/A = {r_mv:.6f}, alpha there = {alpha_mv:.2e}")
print(f"sigma^2_mv = {w_mv @ Sigma @ w_mv:.8f},  1/A = {1 / Acf:.8f}")

assert np.allclose(w_cf, w_kkt, atol=1e-12)
assert abs(alpha - (-lam[0])) < 1e-12
assert abs(dp - alpha) < 1e-7 and abs(w_cf @ Sigma @ w_cf - var_cf) < 1e-12
assert abs(w_mv @ Sigma @ w_mv - 1 / Acf) < 1e-12 and abs(alpha_mv) < 1e-12

A = 21.7918, B = 2.2252, C = 0.2603, D = 0.7215
closed-form w* = [0.2735 0.3624 0.3641]
KKT solve   w* = [0.2735 0.3624 0.3641]   (max diff 3.33e-16)
alpha = 0.238255,  -lambda from KKT = 0.238255
sigma^2(r0) closed form = 0.04776846,  w*^T Sigma w* = 0.04776846
dp*/dr by finite difference = 0.23825503   alpha = 0.23825503   (must agree)
d sigma^2/dr = 0.47651007 = 2 * alpha

minimum-variance return r_mv = B/A = 0.102111, alpha there = 0.00e+00
sigma^2_mv = 0.04588889,  1/A = 0.04588889


The KKT solve reproduces the closed form exactly, the KKT multiplier satisfies $\alpha = -\lambda^\star$,
and the finite-difference derivative of the optimal *half-variance* matches $\alpha$ — confirming the
factor of two: $d\sigma^2/dr = 2\alpha$, not $\alpha$. At $r_{\mathrm{mv}} = B/A$ the multiplier vanishes
and the variance equals $1/A$.

### Problem L2.4: The SVM Written Explicitly as a QP

**Problem Statement:** Write the soft-margin SVM in the canonical QP form
$\min \frac12\mathbf{z}^\top Q\mathbf{z} + \mathbf{c}^\top \mathbf{z}$ s.t. $G\mathbf{z}\le\mathbf{h}$, both in the
primal ($\mathbf{z}=(\mathbf{w},b,\boldsymbol{\xi})$) and in the dual variables, and count variables and
constraints in each. Which form does a solver prefer?

*Intuition:* The primal has one variable per feature, the dual one per example — and only one of them has a $Q$ you can kernelize.

**Solution:**

**Step 1 (primal QP).** With $\mathbf{z} = (\mathbf{w}, b, \boldsymbol{\xi}) \in \mathbb{R}^{n+1+m}$:

$$
\min_{\mathbf{z}}\ \tfrac12\mathbf{z}^\top Q\mathbf{z} + \mathbf{c}^\top \mathbf{z}, \qquad Q = \begin{bmatrix} I_n & 0 & 0 \\ 0 & 0 & 0 \\ 0 & 0 & 0\end{bmatrix} \succeq 0, \qquad \mathbf{c} = \begin{pmatrix} \mathbf{0} \\ 0 \\ C\mathbf{1}_m\end{pmatrix}
$$

subject to the $2m$ inequalities

$$
-y_i\left(\mathbf{w}^\top \mathbf{x}_i + b\right) - \xi_i \le -1, \qquad -\xi_i \le 0
$$

- **Size:** $n + 1 + m$ variables, $2m$ constraints.
- $Q$ is PSD but **singular** (rank $n$): convex, not strictly convex — the bias and slacks are unpenalized.

**Step 2 (dual QP).** With $\mathbf{z} = \boldsymbol{\alpha}\in\mathbb{R}^m$ ([optimization/06](../06_kkt_conditions_and_duality/), Problem L2.4):

$$
\min_{\boldsymbol{\alpha}}\ \tfrac12\boldsymbol{\alpha}^\top \underbrace{\left(\mathbf{y}\mathbf{y}^\top \odot K\right)}_{=:Q_d \succeq 0}\boldsymbol{\alpha} - \mathbf{1}^\top \boldsymbol{\alpha} \quad \text{s.t.}\quad \mathbf{y}^\top \boldsymbol{\alpha} = 0,\ \ \mathbf{0}\le\boldsymbol{\alpha}\le C\mathbf{1}
$$

- **Size:** $m$ variables, $1$ equality plus $2m$ bound constraints.
- $Q_d = \operatorname{diag}(\mathbf{y})\,K\,\operatorname{diag}(\mathbf{y}) \succeq 0$ because $K$ is a
  Gram matrix; congruence preserves positive semidefiniteness.

**Step 3 (which to solve).**

| Regime | Preferred form | Reason |
|---|---|---|
| $n \ll m$ (few features, many examples) | primal | fewer variables; linear SVM solvers (LIBLINEAR) |
| $m \ll n$ or infinite $n$ | dual | size depends only on $m$; kernels available |
| Nonlinear kernel required | dual | $K$ enters directly; the primal $\mathbf{w}$ may be infinite-dimensional |

**Step 4 (structure exploited by real solvers).** The dual has only box constraints plus one equality, so
coordinate-wise updates preserve feasibility if done in *pairs* — this is exactly SMO (sequential minimal
optimization), which optimizes two multipliers at a time analytically and never forms $Q_d$ in full.

**Step 5 (memory).** Forming $K$ costs $O(m^2)$ memory, which is the practical ceiling on kernel SVMs
(around $m \sim 10^5$); beyond that one uses linear SVMs in the primal or random-feature approximations of
$K$.

$$
\boxed{\text{primal: } n{+}1{+}m \ \text{vars}, \ 2m \ \text{constraints}; \quad \text{dual: } m \ \text{vars}, \ \text{box} + \text{1 equality}, \ Q_d = \mathbf{y}\mathbf{y}^\top \odot K}
$$

> **Key takeaway:** Primal and dual are the same QP in different coordinates, but their *sparsity and size* differ completely — choosing between them is the single most consequential modeling decision in kernel machines.

Verification: assemble the primal SVM QP of Step 1 and the dual QP of Step 2 on a tiny separable data set,
solve both with a library optimizer, and confirm the sizes, the equal optimal values, and the support
vectors picked out by complementary slackness.

In [11]:
from scipy.optimize import minimize

Xd = np.array([[2.0, 2.0], [3.0, 3.0], [0.0, 0.0], [1.0, 0.0]])
yd = np.array([1.0, 1.0, -1.0, -1.0])
m_d, n_d = Xd.shape
C = 10.0

# Primal QP in z = (w, b, xi): 0.5 w^T w + C 1^T xi  s.t.  y_i(w^T x_i + b) >= 1 - xi_i, xi >= 0.
def primal_obj(z):
    return 0.5 * z[:n_d] @ z[:n_d] + C * z[n_d + 1:].sum()


primal = minimize(primal_obj, np.zeros(n_d + 1 + m_d), method="SLSQP",
                  constraints=[{"type": "ineq",
                                "fun": lambda z: yd * (Xd @ z[:n_d] + z[n_d]) - 1.0 + z[n_d + 1:]},
                               {"type": "ineq", "fun": lambda z: z[n_d + 1:]}],
                  options={"maxiter": 500, "ftol": 1e-12})
w_p, b_p, xi_p = primal.x[:n_d], primal.x[n_d], primal.x[n_d + 1:]

# Dual QP in alpha: 0.5 alpha^T (yy^T o K) alpha - 1^T alpha  s.t.  y^T alpha = 0, 0 <= alpha <= C.
K = Xd @ Xd.T
Qd = np.outer(yd, yd) * K
dual = minimize(lambda a: 0.5 * a @ Qd @ a - a.sum(), np.zeros(m_d), method="SLSQP",
                constraints=[{"type": "eq", "fun": lambda a: yd @ a}],
                bounds=[(0.0, C)] * m_d, options={"maxiter": 500, "ftol": 1e-14})
alpha = dual.x
w_d = (alpha * yd) @ Xd
sv = np.flatnonzero(alpha > 1e-6)
b_d = float(np.mean(yd[sv] - Xd[sv] @ w_d))

print(f"primal: {n_d + 1 + m_d} variables, {2 * m_d} constraints;  "
      f"dual: {m_d} variables, 1 equality + {2 * m_d} bounds")
print(f"primal  w = {w_p}, b = {b_p:.6f}, xi = {xi_p},  value = {primal.fun:.10f}")
print(f"dual    w = {w_d}, b = {b_d:.6f},               value = {-dual.fun:.10f}")
print(f"Q_d = diag(y) K diag(y) eigenvalues = {np.linalg.eigvalsh(Qd)}  (PSD, rank {np.linalg.matrix_rank(Qd)})")
print(f"alpha = {alpha}  -> support vectors at indices {sv.tolist()}")
print(f"margin 2/||w||_2 = {2 / np.linalg.norm(w_d):.6f}")
assert np.linalg.eigvalsh(Qd)[0] > -1e-9
assert abs(primal.fun + dual.fun) < 1e-6
assert np.abs(w_p - w_d).max() < 1e-6 and abs(b_p - b_d) < 1e-6
assert all(abs(yd[i] * (Xd[i] @ w_d + b_d) - 1.0) < 1e-6 for i in sv)

primal: 7 variables, 8 constraints;  dual: 4 variables, 1 equality + 8 bounds
primal  w = [0.4 0.8], b = -1.400000, xi = [-0. -0. -0. -0.],  value = 0.4000000000
dual    w = [0.4 0.8], b = -1.400000,               value = 0.4000000000
Q_d = diag(y) K diag(y) eigenvalues = [ 0.      0.      0.4904 26.5096]  (PSD, rank 2)
alpha = [0.4 0.  0.  0.4]  -> support vectors at indices [0, 3]
margin 2/||w||_2 = 2.236068


Primal and dual return the same separator $\mathbf{w} = (0.4, 0.8)$, $b = -1.4$ and the same optimal value
$0.4$, so the two QPs of Steps 1 and 2 are the same problem in different coordinates. Only the two
margin-tight points carry $\alpha_i \gt 0$ — the support vectors — and $Q_d$ is PSD but rank deficient, which
is why the dual is convex without being strictly convex.

### Problem L2.5: The Chebyshev Center of a Polyhedron Is an LP

**Problem Statement:** Show that finding the largest inscribed ball in
$P = \{\mathbf{x} : \mathbf{a}_i^\top \mathbf{x} \le b_i,\ i=1,\dots,m\}$ is a linear program. Then compute the
Chebyshev center and radius of the triangle $\{x_1\ge0,\ x_2\ge0,\ x_1+x_2\le1\}$.

*Intuition:* A ball of radius $r$ centred at $\mathbf{x}_c$ fits inside a half-space iff the centre clears the face by $r$ times the normal's length.

**Solution:**

**Step 1 (containment condition for one half-space).** The ball $B(\mathbf{x}_c, r)$ lies in
$\{\mathbf{a}_i^\top \mathbf{x}\le b_i\}$ iff

$$
\sup_{\lVert \mathbf{u}\rVert_2 \le 1}\mathbf{a}_i^\top \left(\mathbf{x}_c + r\mathbf{u}\right) \le b_i \iff \mathbf{a}_i^\top \mathbf{x}_c + r\lVert \mathbf{a}_i\rVert_2 \le b_i
$$

using $\sup_{\lVert \mathbf{u}\rVert\le1}\mathbf{a}^\top \mathbf{u} = \lVert \mathbf{a}\rVert_2$.

**Step 2 (the LP).** The constraint is **affine in $(\mathbf{x}_c, r)$** because
$\lVert \mathbf{a}_i\rVert_2$ is a known constant. Hence

$$
\max_{\mathbf{x}_c,\ r}\ r \quad \text{s.t.}\quad \mathbf{a}_i^\top \mathbf{x}_c + \lVert \mathbf{a}_i\rVert_2\, r \le b_i \ (i=1,\dots,m), \qquad r \ge 0
$$

is a linear program in $n+1$ variables with $m+1$ constraints.

**Step 3 (the triangle).** Write the three faces in the form $\mathbf{a}_i^\top \mathbf{x}\le b_i$:

$$
(-1,0)^\top \mathbf{x} \le 0, \qquad (0,-1)^\top \mathbf{x}\le0, \qquad (1,1)^\top \mathbf{x}\le1
$$

with $\lVert \mathbf{a}_1\rVert = \lVert \mathbf{a}_2\rVert = 1$ and $\lVert \mathbf{a}_3\rVert=\sqrt2$.
The LP is

$$
\max\ r \quad \text{s.t.}\quad -x_1 + r \le 0,\quad -x_2 + r \le 0,\quad x_1 + x_2 + \sqrt2\,r \le 1
$$

**Step 4 (solve).** At the optimum all three constraints bind (otherwise $r$ could increase): the first two
give $x_1 = x_2 = r$, and substituting into the third,

$$
2r + \sqrt2\,r = 1 \ \Longrightarrow\ r^* = \frac{1}{2+\sqrt2} = \frac{2-\sqrt2}{2} \approx 0.2929
$$

$$
\mathbf{x}_c^* = \left(0.2929,\ 0.2929\right)
$$

**Step 5 (cross-check with geometry).** For a right triangle with legs $a=b=1$ and hypotenuse $c=\sqrt2$,
the inradius is $r = \frac{a+b-c}{2} = \frac{2-\sqrt2}{2}$ $\checkmark$, and the incentre of a right
triangle lies at distance $r$ from both legs $\checkmark$.

$$
\boxed{\max r \ \text{s.t.}\ \mathbf{a}_i^\top \mathbf{x}_c + \lVert \mathbf{a}_i\rVert_2 r \le b_i \ \text{is an LP}; \ \text{triangle: } \mathbf{x}_c = (0.2929, 0.2929),\ r = \tfrac{2-\sqrt2}{2}}
$$

> **Key takeaway:** A robustness requirement ("the point must stay feasible under any perturbation of size $r$") turned into an ordinary linear constraint — the *simplest* instance of robust optimization, and the template for the SOCP of Problem L2.6.

Verification: build the Chebyshev-centre LP of Step 2 for the triangle, solve it, and cross-check against
the inradius formula $r = (a + b - c)/2$.

In [12]:
A_cb = np.array([[-1.0, 0.0], [0.0, -1.0], [1.0, 1.0]])
b_cb = np.array([0.0, 0.0, 1.0])
norms = np.linalg.norm(A_cb, axis=1)

A_lp = np.hstack([A_cb, norms.reshape(-1, 1)])   # a_i^T xc + ||a_i|| r <= b_i
c_lp = np.array([0.0, 0.0, -1.0])                # maximize r
cheb = linprog(c_lp, A_ub=A_lp, b_ub=b_cb,
               bounds=[(None, None), (None, None), (0, None)], method="highs")
xc, r_star = cheb.x[:2], cheb.x[2]
r_formula = (1.0 + 1.0 - np.sqrt(2.0)) / 2.0

print(f"||a_i||_2 = {norms}")
print(f"Chebyshev centre = {xc},  radius r* = {r_star:.10f}")
print(f"inradius (a+b-c)/2 for legs 1, 1     = {r_formula:.10f}")
print(f"|difference|                          = {abs(r_star - r_formula):.3e}")
print(f"slack of every face at the optimum   = {b_cb - A_lp @ cheb.x} (all zero: all three bind)")
assert abs(r_star - r_formula) < 1e-9 and np.allclose(xc, [r_formula, r_formula], atol=1e-9)

||a_i||_2 = [1.     1.     1.4142]
Chebyshev centre = [0.2929 0.2929],  radius r* = 0.2928932188
inradius (a+b-c)/2 for legs 1, 1     = 0.2928932188
|difference|                          = 5.551e-17
slack of every face at the optimum   = [0. 0. 0.] (all zero: all three bind)


The LP returns $r^\star = (2-\sqrt2)/2 \approx 0.2929$ with centre $(r^\star, r^\star)$, matching the
inradius formula to $10^{-9}$ and confirming that all three faces are tight at the optimum.

### Problem L2.6: Robust Least Squares Is an SOCP

**Problem Statement:** Consider the worst-case problem
$\min_{\mathbf{x}}\max_{\lVert \Delta\rVert_2 \le \rho}\lVert (A+\Delta)\mathbf{x}-\mathbf{b}\rVert_2$,
where $\lVert \cdot\rVert_2$ on $\Delta$ is the spectral norm. Prove the inner maximum equals
$\lVert A\mathbf{x}-\mathbf{b}\rVert_2 + \rho\lVert \mathbf{x}\rVert_2$, write the result as an SOCP, and
relate it to Tikhonov regularization.

*Intuition:* The adversary aligns its perturbation with the residual direction, adding exactly $\rho\lVert \mathbf{x}\rVert$ of extra error.

**Solution:**

**Step 1 (upper bound).** By the triangle inequality and $\lVert \Delta\mathbf{x}\rVert \le \lVert \Delta\rVert_2\lVert \mathbf{x}\rVert$:

$$
\lVert (A+\Delta)\mathbf{x}-\mathbf{b}\rVert \le \lVert A\mathbf{x}-\mathbf{b}\rVert + \rho\lVert \mathbf{x}\rVert
$$

**Step 2 (the bound is attained).** Let $\mathbf{r} = A\mathbf{x}-\mathbf{b}$. Assume
$\mathbf{r}\neq\mathbf{0}$ and $\mathbf{x}\neq\mathbf{0}$, and choose the rank-one perturbation

$$
\Delta^\star = \rho\,\frac{\mathbf{r}}{\lVert \mathbf{r}\rVert}\,\frac{\mathbf{x}^\top }{\lVert \mathbf{x}\rVert}, \qquad \lVert \Delta^\star\rVert_2 = \rho
$$

Then $\Delta^\star\mathbf{x} = \rho\lVert \mathbf{x}\rVert\frac{\mathbf{r}}{\lVert \mathbf{r}\rVert}$ is
parallel to $\mathbf{r}$, so

$$
\lVert (A+\Delta^\star)\mathbf{x}-\mathbf{b}\rVert = \left\lVert \mathbf{r}\left(1 + \frac{\rho\lVert \mathbf{x}\rVert}{\lVert \mathbf{r}\rVert}\right)\right\rVert = \lVert \mathbf{r}\rVert + \rho\lVert \mathbf{x}\rVert
$$

(The degenerate cases $\mathbf{r} = \mathbf{0}$ or $\mathbf{x}=\mathbf{0}$ are handled by choosing any
unit direction.) Hence the max equals the bound.

**Step 3 (the resulting convex problem).**

$$
\min_{\mathbf{x}}\ \lVert A\mathbf{x}-\mathbf{b}\rVert_2 + \rho\lVert \mathbf{x}\rVert_2
$$

a sum of two Euclidean norms — convex, but nonsmooth at $\mathbf{x} = \mathbf{0}$ and where the residual
vanishes.

**Step 4 (SOCP form).** Introduce epigraph variables $t_1, t_2$:

$$
\min_{\mathbf{x},t_1,t_2}\ t_1 + \rho t_2 \quad \text{s.t.}\quad \lVert A\mathbf{x}-\mathbf{b}\rVert_2 \le t_1, \quad \lVert \mathbf{x}\rVert_2 \le t_2
$$

a linear objective over two second-order cone constraints: an SOCP with $n+2$ variables.

**Step 5 (contrast with Tikhonov).** Ridge regression minimizes
$\lVert A\mathbf{x}-\mathbf{b}\rVert_2^2 + \lambda\lVert \mathbf{x}\rVert_2^2$ — a sum of *squares*, which
is a QP with a closed form. Robust least squares minimizes a sum of *norms*: it is not a QP, its solution
is not linear in $\mathbf{b}$, and it produces exactly-zero solutions for $\rho$ large enough (the
nonsmooth kink at $\mathbf{x}=\mathbf{0}$), which the ridge penalty never does. The two coincide only in
spirit: both shrink $\mathbf{x}$, and both can be derived as worst-case or Bayesian responses to data
uncertainty.

$$
\boxed{\max_{\lVert \Delta\rVert_2\le\rho}\lVert (A+\Delta)\mathbf{x}-\mathbf{b}\rVert = \lVert A\mathbf{x}-\mathbf{b}\rVert + \rho\lVert \mathbf{x}\rVert; \ \text{an SOCP in } (\mathbf{x},t_1,t_2)}
$$

> **Key takeaway:** Robustness is a *modeling* choice with a precise algebraic price: the worst case over an ellipsoidal uncertainty set turns a QP into an SOCP, one level up the hierarchy but still solvable to global optimality.

Verification: sample $4000$ spectral-norm-bounded perturbations $\Delta$ and confirm that none beats
$\lVert A\mathbf{x}-\mathbf{b}\rVert_2 + \rho\lVert \mathbf{x}\rVert_2$, while the rank-one $\Delta^\star$
of Step 2 attains it.

In [13]:
A_r = rng.normal(size=(5, 3))
b_r = rng.normal(size=5)
x_r = rng.normal(size=3)
rho = 0.4

resid = A_r @ x_r - b_r
bound = np.linalg.norm(resid) + rho * np.linalg.norm(x_r)

sampled = 0.0
for _ in range(4000):
    D = rng.normal(size=(5, 3))
    D = rho * D / np.linalg.norm(D, 2)
    sampled = max(sampled, np.linalg.norm((A_r + D) @ x_r - b_r))

D_star = rho * np.outer(resid / np.linalg.norm(resid), x_r / np.linalg.norm(x_r))
attained = np.linalg.norm((A_r + D_star) @ x_r - b_r)

print(f"||A x - b|| + rho ||x||        = {bound:.10f}")
print(f"best of 4000 random Delta      = {sampled:.10f}   (must be <= the bound)")
print(f"value at the rank-one Delta*   = {attained:.10f}   (must equal the bound)")
print(f"spectral norm ||Delta*||_2     = {np.linalg.norm(D_star, 2):.6f}   (must equal rho = {rho})")
assert sampled <= bound + 1e-12
assert abs(attained - bound) < 1e-10 and abs(np.linalg.norm(D_star, 2) - rho) < 1e-10

||A x - b|| + rho ||x||        = 1.7820698125


best of 4000 random Delta      = 1.7777943126   (must be <= the bound)
value at the rank-one Delta*   = 1.7820698125   (must equal the bound)
spectral norm ||Delta*||_2     = 0.400000   (must equal rho = 0.4)


No random perturbation exceeds the bound, the explicit rank-one $\Delta^\star$ attains it exactly, and its
spectral norm equals $\rho$ — so the inner maximum really is $\lVert A\mathbf{x}-\mathbf{b}\rVert_2 + \rho\lVert \mathbf{x}\rVert_2$
and the SOCP of Step 4 is an exact reformulation.

## L3 — Challenge Proofs

### Problem L3.1: Extreme Points Are Exactly Basic Feasible Solutions

**Problem Statement:** For $P = \{\mathbf{x}\in\mathbb{R}^n : A\mathbf{x}=\mathbf{b},\ \mathbf{x}\ge\mathbf{0}\}$
with $A\in\mathbb{R}^{m\times n}$, prove that $\mathbf{x}\in P$ is an extreme point of $P$ if and only if
the columns $\{A_j : x_j \gt 0\}$ are linearly independent (i.e. $\mathbf{x}$ is a basic feasible
solution).

*Intuition:* A dependency among the active columns is a direction you can move both ways while staying feasible — which is exactly what an extreme point forbids.

**Solution:**

Let $S = \{j : x_j \gt 0\}$ be the support of $\mathbf{x}$.

**Step 1 ($\Leftarrow$: independence implies extreme).** Suppose $\{A_j\}_{j\in S}$ is independent and
$\mathbf{x} = \theta\mathbf{y} + (1-\theta)\mathbf{z}$ with $\mathbf{y},\mathbf{z}\in P$,
$\theta\in(0,1)$. For $j \notin S$, $x_j = 0$ and $y_j, z_j \ge 0$ force $y_j = z_j = 0$: both points share
the support restriction. Then

$$
A\left(\mathbf{y}-\mathbf{z}\right) = \mathbf{b}-\mathbf{b} = \mathbf{0}, \qquad \text{with } (\mathbf{y}-\mathbf{z})_j = 0 \ \text{for } j\notin S
$$

so $\sum_{j\in S}(y_j - z_j)A_j = \mathbf{0}$. Independence forces $y_j = z_j$ for all $j\in S$, hence
$\mathbf{y}=\mathbf{z}=\mathbf{x}$: $\mathbf{x}$ is extreme.

**Step 2 ($\Rightarrow$: dependence implies not extreme).** Suppose $\{A_j\}_{j\in S}$ is dependent: there
are scalars $d_j$, not all zero, with $\sum_{j\in S}d_jA_j = \mathbf{0}$. Extend $\mathbf{d}$ by
$d_j = 0$ for $j\notin S$, so $A\mathbf{d} = \mathbf{0}$ and $\mathbf{d}\neq\mathbf{0}$.

**Step 3 (construct the two points).** Since $x_j \gt 0$ for all $j\in S$ and $S$ is finite, choose

$$
\epsilon = \min_{j \in S,\ d_j \neq 0}\frac{x_j}{\lvert d_j\rvert} \gt 0
$$

and set $\mathbf{y} = \mathbf{x}+\epsilon\mathbf{d}$, $\mathbf{z} = \mathbf{x}-\epsilon\mathbf{d}$. Then:

- $A\mathbf{y} = A\mathbf{x} + \epsilon A\mathbf{d} = \mathbf{b}$, likewise for $\mathbf{z}$.
- For $j\in S$: $\lvert \epsilon d_j\rvert \le x_j$ by the choice of $\epsilon$, so $y_j, z_j \ge 0$.
- For $j\notin S$: $y_j = z_j = 0 \ge 0$.

So $\mathbf{y},\mathbf{z}\in P$, they are distinct ($\mathbf{d}\neq\mathbf{0}$), and
$\mathbf{x} = \frac12\mathbf{y}+\frac12\mathbf{z}$: $\mathbf{x}$ is **not** extreme.

**Step 4 (consequences).** Since $\{A_j\}_{j\in S}$ independent forces $\lvert S\rvert \le \operatorname{rank}(A) \le m$,
every extreme point has at most $m$ nonzero coordinates, and there are at most $\binom{n}{m}$ of them —
finitely many. Together with Problem L0.1 this proves the fundamental theorem of LP and bounds the simplex
search space.

$$
\boxed{\mathbf{x} \ \text{extreme point of } P \iff \{A_j : x_j \gt 0\} \ \text{linearly independent} \iff \mathbf{x} \ \text{is a BFS}}
$$

> **Key takeaway:** The bridge between geometry (extreme points) and algebra (independent columns) is what lets an algorithm move on a polyhedron by *swapping columns of a matrix* — the pivot operation of the simplex method.

Verification: run the *construction* of Steps 2-3 on a small standard-form polyhedron. For each basic
feasible solution the support columns admit no nontrivial null vector, so no splitting exists; for a
non-basic feasible point the null vector produces two distinct feasible points whose midpoint it is.

In [14]:
A_e = np.array([[1.0, 1.0, 1.0, 0.0],
                [1.0, 3.0, 0.0, 1.0]])
b_e = np.array([4.0, 6.0])


def null_space(M):
    """Orthonormal basis of the null space of M, as rows."""
    _, sv, Vt = np.linalg.svd(M)
    tol = max(M.shape) * np.finfo(float).eps * (sv[0] if sv.size else 1.0)
    return Vt[np.sum(sv > tol):]


def try_split(x, A, b):
    """Return (y, z) with x = (y+z)/2, y != z, both feasible, or None if the construction fails."""
    S = np.flatnonzero(x > 1e-12)
    N = null_space(A[:, S])
    if N.shape[0] == 0:
        return None
    d = np.zeros(len(x))
    d[S] = N[0]
    eps = min(x[j] / abs(d[j]) for j in S if abs(d[j]) > 1e-12)
    return x + eps * d, x - eps * d


for B in itertools.combinations(range(4), 2):
    AB = A_e[:, list(B)]
    if abs(np.linalg.det(AB)) < 1e-12:
        continue
    xB = np.linalg.solve(AB, b_e)
    if (xB < -1e-12).any():
        continue
    x = np.zeros(4)
    x[list(B)] = xB
    S = np.flatnonzero(x > 1e-12)
    independent = np.linalg.matrix_rank(A_e[:, S]) == len(S)
    split = try_split(x, A_e, b_e)
    print(f"x = {x}  support {S.tolist()}  independent = {independent}  splittable = {split is not None}")
    assert independent and split is None

x_mid = 0.5 * np.array([3.0, 1.0, 0.0, 0.0]) + 0.5 * np.array([0.0, 0.0, 4.0, 6.0])
S_mid = np.flatnonzero(x_mid > 1e-12)
y_s, z_s = try_split(x_mid, A_e, b_e)
print(f"\nnon-basic point x = {x_mid}  support size {len(S_mid)} > rank(A) = "
      f"{np.linalg.matrix_rank(A_e)}  -> dependent")
print(f"  y = {y_s}  feasible = {bool(np.allclose(A_e @ y_s, b_e) and (y_s >= -1e-12).all())}")
print(f"  z = {z_s}  feasible = {bool(np.allclose(A_e @ z_s, b_e) and (z_s >= -1e-12).all())}")
print(f"  midpoint residual ||(y+z)/2 - x|| = {np.linalg.norm(0.5 * (y_s + z_s) - x_mid):.2e}")
assert np.allclose(A_e @ y_s, b_e) and (y_s >= -1e-12).all()
assert np.allclose(A_e @ z_s, b_e) and (z_s >= -1e-12).all()
assert np.linalg.norm(y_s - z_s) > 1e-6
assert np.allclose(0.5 * (y_s + z_s), x_mid)

x = [3. 1. 0. 0.]  support [0, 1]  independent = True  splittable = False
x = [4. 0. 0. 2.]  support [0, 3]  independent = True  splittable = False
x = [0. 2. 2. 0.]  support [1, 2]  independent = True  splittable = False
x = [0. 0. 4. 6.]  support [2, 3]  independent = True  splittable = False

non-basic point x = [1.5 0.5 2.  3. ]  support size 4 > rank(A) = 2  -> dependent
  y = [0.     0.9223 3.0777 3.233 ]  feasible = True
  z = [3.     0.0777 0.9223 2.767 ]  feasible = True
  midpoint residual ||(y+z)/2 - x|| = 0.00e+00


Every basic feasible solution resists the construction — its support columns have trivial null space, so
Step 1's argument leaves no room to move — while the non-basic point is split explicitly into two feasible
neighbours of which it is the midpoint, exactly as Steps 2-3 prescribe.

### Problem L3.2: Chebyshev Approximation and Its Dual

**Problem Statement:** Show that $\min_{\mathbf{x}}\lVert A\mathbf{x}-\mathbf{b}\rVert_\infty$ is an LP,
derive its dual, and interpret the dual constraint $\lVert \mathbf{y}\rVert_1 \le 1$. Compare with least
squares and $\ell_1$ regression.

*Intuition:* Minimizing a max becomes minimizing one variable that dominates every residual — and dualizing a max-norm produces its dual norm, the $\ell_1$.

**Solution:**

**Step 1 (epigraph reformulation).** $\lVert \mathbf{r}\rVert_\infty \le t$ means
$-t \le r_i \le t$ for every $i$, so

$$
\min_{\mathbf{x},t}\ t \quad \text{s.t.}\quad A\mathbf{x}-\mathbf{b} \le t\mathbf{1}, \quad -\left(A\mathbf{x}-\mathbf{b}\right) \le t\mathbf{1}
$$

an LP in $n+1$ variables with $2m$ constraints.

**Step 2 (Lagrangian).** With multipliers $\mathbf{u},\mathbf{v}\ge\mathbf{0}$ for the two blocks,

$$
\mathcal{L} = t + \mathbf{u}^\top \left(A\mathbf{x}-\mathbf{b}-t\mathbf{1}\right) + \mathbf{v}^\top \left(\mathbf{b}-A\mathbf{x}-t\mathbf{1}\right)
$$

$$
= t\left(1 - \mathbf{1}^\top \mathbf{u} - \mathbf{1}^\top \mathbf{v}\right) + \mathbf{x}^\top A^\top \left(\mathbf{u}-\mathbf{v}\right) + \mathbf{b}^\top \left(\mathbf{v}-\mathbf{u}\right)
$$

**Step 3 (dual function).** Minimizing over the free variables $t\in\mathbb{R}$ and
$\mathbf{x}\in\mathbb{R}^n$ gives $-\infty$ unless both linear coefficients vanish:

$$
\mathbf{1}^\top \left(\mathbf{u}+\mathbf{v}\right) = 1, \qquad A^\top \left(\mathbf{u}-\mathbf{v}\right) = \mathbf{0}
$$

and then $g = \mathbf{b}^\top (\mathbf{v}-\mathbf{u})$. Substituting $\mathbf{y} = \mathbf{u}-\mathbf{v}$ and
noting that for optimal $\mathbf{u},\mathbf{v}\ge0$ one may take
$\mathbf{u} = (\mathbf{y})_+$, $\mathbf{v} = (\mathbf{y})_-$ so that
$\mathbf{1}^\top (\mathbf{u}+\mathbf{v}) = \lVert \mathbf{y}\rVert_1$, the dual is

$$
\max_{\mathbf{y}}\ -\mathbf{b}^\top \mathbf{y} \quad \text{s.t.}\quad A^\top \mathbf{y} = \mathbf{0}, \quad \lVert \mathbf{y}\rVert_1 \le 1
$$

(equivalently $\max\ \mathbf{b}^\top \mathbf{y}$ with the sign of $\mathbf{y}$ flipped).

**Step 4 (interpretation).** The dual searches for a direction $\mathbf{y}$ **orthogonal to the column
space of $A$** (so it cannot be explained by any $\mathbf{x}$) that correlates maximally with $\mathbf{b}$,
measured with the unit $\ell_1$ ball — the dual ball of $\ell_\infty$. This is the general pattern:

$$
\left(\ell_\infty\right)^* = \ell_1, \qquad \left(\ell_1\right)^* = \ell_\infty, \qquad \left(\ell_2\right)^* = \ell_2
$$

**Step 5 (comparison of the three fits).**

| Loss | Problem class | Behaviour |
|---|---|---|
| $\lVert A\mathbf{x}-\mathbf{b}\rVert_2$ | QP / linear algebra | smooth, closed form, sensitive to outliers |
| $\lVert A\mathbf{x}-\mathbf{b}\rVert_1$ | LP | robust to outliers, median-like, sparse residuals |
| $\lVert A\mathbf{x}-\mathbf{b}\rVert_\infty$ | LP | minimax; equioscillation, extremely outlier-sensitive |

Complementary slackness in the $\ell_\infty$ dual says $y_i \neq 0$ only where the residual attains the
maximum $t^*$ — the **equioscillation** property behind Chebyshev polynomial approximation and Remez's
algorithm.

$$
\boxed{\min_{\mathbf{x},t}\ t \ \text{s.t.}\ \lVert A\mathbf{x}-\mathbf{b}\rVert_\infty \le t
\quad \Longleftrightarrow \quad
\max_{\mathbf{y}}\ -\mathbf{b}^\top \mathbf{y}\ \text{s.t.}\ A^\top \mathbf{y}=\mathbf{0},\ \lVert \mathbf{y}\rVert_1\le 1,
\ \text{equioscillation at the tight residuals}}
$$

> **Key takeaway:** Dualizing a norm-minimization problem always produces the *dual norm* as a constraint; this single fact organizes $\ell_1$/$\ell_\infty$ duality, LASSO dual certificates, and the geometry of minimax approximation.

Verification: solve the Chebyshev LP of Step 1 on a small design matrix, solve the dual of Step 3, and read
off the equioscillation predicted by complementary slackness.

In [15]:
A_ch = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0], [1.0, 3.0]])
b_ch = np.array([1.0, 0.0, 1.5, 1.0])
m_ch, n_ch = A_ch.shape

A_ub = np.vstack([np.hstack([A_ch, -np.ones((m_ch, 1))]),
                  np.hstack([-A_ch, -np.ones((m_ch, 1))])])
b_ub = np.concatenate([b_ch, -b_ch])
prim = linprog(np.array([0.0, 0.0, 1.0]), A_ub=A_ub, b_ub=b_ub,
               bounds=[(None, None)] * n_ch + [(0, None)], method="highs")
x_ch, t_ch = prim.x[:n_ch], prim.x[n_ch]
resid = A_ch @ x_ch - b_ch

# dual: max -b^T y  s.t.  A^T y = 0, ||y||_1 <= 1, written with y = p - q, p, q >= 0
dual = linprog(np.concatenate([b_ch, -b_ch]),
               A_ub=np.ones((1, 2 * m_ch)), b_ub=np.array([1.0]),
               A_eq=np.hstack([A_ch.T, -A_ch.T]), b_eq=np.zeros(n_ch),
               bounds=[(0, None)] * (2 * m_ch), method="highs")
y_ch = dual.x[:m_ch] - dual.x[m_ch:]

print(f"x* = {x_ch},  t* = ||Ax-b||_inf = {t_ch:.6f}")
print(f"residuals            = {resid}")
print(f"dual y*              = {y_ch}   ||y*||_1 = {np.abs(y_ch).sum():.6f}   A^T y* = {A_ch.T @ y_ch}")
print(f"dual objective -b^T y* = {-b_ch @ y_ch:.6f}   (equals t*)")
print(f"|residual| = t* where y_i != 0 : "
      f"{[bool(abs(abs(resid[i]) - t_ch) < 1e-9) for i in range(m_ch) if abs(y_ch[i]) > 1e-9]}")
assert abs(np.abs(resid).max() - t_ch) < 1e-9
assert abs(-b_ch @ y_ch - t_ch) < 1e-9 and np.abs(A_ch.T @ y_ch).max() < 1e-9
assert np.abs(y_ch).sum() <= 1.0 + 1e-9
for i in range(m_ch):
    if abs(y_ch[i]) > 1e-9:
        assert abs(abs(resid[i]) - t_ch) < 1e-9

x* = [0.375 0.25 ],  t* = ||Ax-b||_inf = 0.625000
residuals            = [-0.625  0.625 -0.625  0.125]
dual y*              = [-0.25  0.5  -0.25  0.  ]   ||y*||_1 = 1.000000   A^T y* = [0. 0.]
dual objective -b^T y* = 0.625000   (equals t*)
|residual| = t* where y_i != 0 : [True, True, True]


The LP gives $t^\star = 0.625$, the dual attains the same value with $\lVert \mathbf{y}^\star\rVert_1 = 1$
and $A^\top \mathbf{y}^\star = \mathbf{0}$, and every index with $y_i^\star \ne 0$ has $\lvert r_i\rvert = t^\star$
with alternating signs — equioscillation, produced by complementary slackness rather than assumed.

### Problem L3.3: The Max-Cut SDP Relaxation on a Triangle

**Problem Statement:** For the unweighted triangle $K_3$, write the Goemans-Williamson SDP relaxation

$$
\max\ \frac12\sum_{i \lt j}w_{ij}\left(1 - X_{ij}\right) \quad \text{s.t.}\quad X \succeq 0,\ X_{ii} = 1
$$

solve it exactly using symmetry, compare with the true max-cut value, and compute the integrality ratio.

*Intuition:* On a symmetric graph the optimal $X$ can be taken symmetric, which collapses the SDP to a one-parameter problem.

**Solution:**

**Step 1 (the relaxation).** Max-cut is
$\max_{\mathbf{s}\in\{\pm1\}^3}\frac12\sum_{i \lt j}(1-s_is_j)$. Replacing $s_i$ by unit vectors
$\mathbf{v}_i$ and setting $X_{ij} = \mathbf{v}_i^\top \mathbf{v}_j$ gives $X\succeq0$ with unit diagonal:

$$
\text{SDP} = \max\ \frac12\left[3 - \left(X_{12}+X_{13}+X_{23}\right)\right] \quad \text{s.t.}\quad X\succeq0,\ X_{11}=X_{22}=X_{33}=1
$$

so we must **minimize** $s = X_{12}+X_{13}+X_{23}$.

**Step 2 (symmetrize).** If $X$ is feasible, so is $P^\top XP$ for any permutation matrix $P$ (the constraints
and the objective are permutation invariant), and the feasible set is convex. Averaging $X$ over all $3!$
permutations therefore yields a feasible matrix with the same objective value and the symmetric form

$$
X(t) = (1-t)I + tJ, \qquad J = \mathbf{1}\mathbf{1}^\top $$

so we may restrict to this one-parameter family without loss of optimality.

**Step 3 (PSD range of $t$).** The eigenvalues of $X(t)$ are

$$
1 + 2t \ \text{(eigenvector } \mathbf{1}\text{)}, \qquad 1 - t \ \text{(multiplicity 2)}
$$

so $X(t)\succeq0 \iff -\tfrac12 \le t \le 1$.

**Step 4 (optimize).** Here $s = 3t$, minimized at $t = -\tfrac12$, giving $s_{\min} = -\tfrac32$ and

$$
\text{SDP}^* = \frac12\left(3 + \frac32\right) = \frac94 = 2.25
$$

The optimal $X = \frac32 I - \frac12 J$ is realized by three unit vectors at mutual angles $120^\circ$ in
$\mathbb{R}^2$ — the geometric picture of the relaxation "spreading the triangle out as much as possible".

**Step 5 (true max-cut and the ratio).** Any $\pm1$ assignment on a triangle must put two vertices on the
same side, so exactly $2$ of the $3$ edges are cut:

$$
\text{MAXCUT}(K_3) = 2
$$

The integrality ratio is

$$
\frac{\text{MAXCUT}}{\text{SDP}^*} = \frac{2}{9/4} = \frac{8}{9} \approx 0.8889
$$

comfortably above the Goemans-Williamson guarantee $\alpha_{GW} \approx 0.87856$ (whose worst case is
approached by the $5$-cycle, ratio $\approx 0.8845$). The gap is entirely due to *frustration*: the
triangle is an odd cycle, so no $\pm1$ labelling can make all three edges disagree, while unit vectors at
$120^\circ$ can.

$$
\boxed{\text{SDP}^* = \tfrac94, \quad \text{MAXCUT} = 2, \quad \text{ratio} = \tfrac89 \approx 0.889 \gt \alpha_{GW} \approx 0.8786}
$$

> **Key takeaway:** The SDP relaxation lifts $\pm1$ scalars to unit vectors, which buys convexity at the price of an integrality gap; the entire Goemans-Williamson analysis is the statement that random hyperplane rounding recovers at least $87.856\%$ of the lifted value.

Verification: confirm $\text{SDP}^\star = 9/4$ by a grid search over unit-diagonal PSD matrices (no
symmetry assumed), and compute the true max-cut by brute force over the eight labellings.

In [16]:
X_opt = 1.5 * np.eye(3) - 0.5 * np.ones((3, 3))
print(f"symmetrized X = 1.5 I - 0.5 J:\n{X_opt}")
print(f"eigenvalues {np.linalg.eigvalsh(X_opt)}, diagonal {np.diag(X_opt)}")
print(f"objective 0.5*(3 - sum of off-diagonals) = "
      f"{0.5 * (3 - (X_opt[0, 1] + X_opt[0, 2] + X_opt[1, 2])):.6f}")

grid = np.linspace(-1.0, 1.0, 161)
best_val, best_triple = -np.inf, None
for a in grid:
    for bb in grid:
        for cc in grid:
            M = np.array([[1.0, a, bb], [a, 1.0, cc], [bb, cc, 1.0]])
            if np.linalg.eigvalsh(M)[0] >= -1e-12:
                v = 0.5 * (3.0 - (a + bb + cc))
                if v > best_val:
                    best_val, best_triple = v, (a, bb, cc)
print(f"\ngrid search over PSD unit-diagonal X: best = {best_val:.6f} at {np.round(best_triple, 4)}")

cut_best = max(sum(1 for i, j in [(0, 1), (0, 2), (1, 2)] if s[i] != s[j])
               for s in itertools.product([-1, 1], repeat=3))
print(f"true MAXCUT(K3) by brute force        = {cut_best}")
print(f"integrality ratio MAXCUT / SDP*       = {cut_best / 2.25:.6f} = 8/9")
print(f"Goemans-Williamson guarantee          = 0.878567")
assert abs(best_val - 2.25) < 1e-6 and cut_best == 2
assert abs(cut_best / 2.25 - 8 / 9) < 1e-12 and cut_best / 2.25 > 0.878567

symmetrized X = 1.5 I - 0.5 J:
[[ 1.  -0.5 -0.5]
 [-0.5  1.  -0.5]
 [-0.5 -0.5  1. ]]
eigenvalues [-0.   1.5  1.5], diagonal [1. 1. 1.]
objective 0.5*(3 - sum of off-diagonals) = 2.250000



grid search over PSD unit-diagonal X: best = 2.250000 at [-0.5 -0.5 -0.5]
true MAXCUT(K3) by brute force        = 2
integrality ratio MAXCUT / SDP*       = 0.888889 = 8/9
Goemans-Williamson guarantee          = 0.878567


The grid search over *all* unit-diagonal PSD matrices peaks at $9/4$, so the symmetrization argument of
Step 2 lost nothing; brute force gives $\text{MAXCUT}(K_3) = 2$ and the ratio $8/9 \approx 0.8889$, safely
above the Goemans-Williamson constant.

### Problem L3.4: Farkas' Lemma Implies LP Strong Duality

**Problem Statement:** State Farkas' lemma and use it to prove LP strong duality: if
$p^* = \min\{\mathbf{c}^\top \mathbf{x} : A\mathbf{x}\ge\mathbf{b},\ \mathbf{x}\ge\mathbf{0}\}$ is finite, then
the dual optimum $d^* = \max\{\mathbf{b}^\top \mathbf{y} : A^\top \mathbf{y}\le\mathbf{c},\ \mathbf{y}\ge\mathbf{0}\}$
equals $p^*$ and is attained.

*Intuition:* Farkas says a linear system either has a solution or has a certificate of infeasibility — apply it to the system "beat the optimum".

**Solution:**

**Step 1 (Farkas' lemma).** For $M\in\mathbb{R}^{k\times\ell}$ and $\mathbf{q}\in\mathbb{R}^k$, exactly one
of the following holds:

1. $\exists\,\mathbf{z}\ge\mathbf{0}$ with $M\mathbf{z} = \mathbf{q}$;
2. $\exists\,\mathbf{w}$ with $M^\top \mathbf{w}\le\mathbf{0}$ and $\mathbf{q}^\top \mathbf{w} \gt 0$.

(Geometrically: either $\mathbf{q}$ lies in the finitely generated cone spanned by the columns of $M$, or a
hyperplane separates it from that closed cone.) The variant used below is the inequality form: exactly one
of $\{A\mathbf{x}\ge\mathbf{b},\ \mathbf{x}\ge\mathbf{0}\}$ solvable, or
$\{A^\top \mathbf{y}\le\mathbf{0},\ \mathbf{y}\ge\mathbf{0},\ \mathbf{b}^\top \mathbf{y} \gt 0\}$ solvable.

**Step 2 (set up the "no better than $p^*$" system).** By definition of $p^*$, the system

$$
A\mathbf{x} \ge \mathbf{b}, \qquad -\mathbf{c}^\top \mathbf{x} \ge -p^* + \epsilon, \qquad \mathbf{x}\ge\mathbf{0}
$$

is **infeasible** for every $\epsilon \gt 0$ (a solution would beat the optimum). Stack it as
$\tilde{A}\mathbf{x}\ge\tilde{\mathbf{b}}$ with

$$
\tilde{A} = \begin{bmatrix} A \\ -\mathbf{c}^\top \end{bmatrix}, \qquad \tilde{\mathbf{b}} = \begin{pmatrix} \mathbf{b} \\ -p^*+\epsilon\end{pmatrix}
$$

**Step 3 (apply Farkas).** Infeasibility gives $(\mathbf{y}, \sigma) \ge \mathbf{0}$ with

$$
A^\top \mathbf{y} - \sigma\mathbf{c} \le \mathbf{0}, \qquad \mathbf{b}^\top \mathbf{y} + \sigma\left(-p^*+\epsilon\right) \gt 0
$$

**Step 4 (rule out $\sigma = 0$).** If $\sigma = 0$ then $A^\top \mathbf{y}\le\mathbf{0}$, $\mathbf{y}\ge0$ and
$\mathbf{b}^\top \mathbf{y} \gt 0$, which by Farkas again would make the *primal* infeasible — contradicting
that $p^*$ is finite (hence the primal is feasible). So $\sigma \gt 0$, and we may normalize
$\sigma = 1$ by rescaling $(\mathbf{y},\sigma)$.

**Step 5 (conclude).** With $\sigma = 1$ we obtain $\mathbf{y}\ge\mathbf{0}$ with

$$
A^\top \mathbf{y} \le \mathbf{c} \quad (\text{dual feasible}), \qquad \mathbf{b}^\top \mathbf{y} \gt p^* - \epsilon
$$

so $d^* \ge p^* - \epsilon$ for every $\epsilon \gt 0$, giving $d^* \ge p^*$. Weak duality
([optimization/06](../06_kkt_conditions_and_duality/), Problem L0.3) gives $d^*\le p^*$. Hence $d^* = p^*$. Attainment follows because the dual
feasible region is a polyhedron on which a bounded linear objective attains its maximum at a vertex
(Problem L0.1).

$$
\boxed{p^* \ \text{finite} \implies d^* = p^* \ \text{and the dual optimum is attained (LP strong duality)}}
$$

> **Key takeaway:** LP duality needs no Slater condition because Farkas' lemma is a *theorem of the alternative* about polyhedra — finitely generated cones are automatically closed, which is exactly what fails for general convex cones and allows the gaps discussed in [optimization/06](../06_kkt_conditions_and_duality/), Problem L3.3.

Verification: on the LP of Problem L1.2, exhibit the Farkas certificate of Step 4 for the infeasible
"beat the optimum" system, and confirm that the dual optimum equals $p^\star$ and is attained at a vertex.

In [17]:
c_f = np.array([4.0, 3.0])
A_f = np.array([[1.0, 1.0], [2.0, 1.0]])
b_f = np.array([3.0, 4.0])
p_star = linprog(c_f, A_ub=-A_f, b_ub=-b_f, bounds=[(0, None)] * 2, method="highs").fun

for eps in [1e-3, 1e-2, 1e-1]:
    tight = linprog(np.zeros(2),
                    A_ub=np.vstack([-A_f, c_f]), b_ub=np.concatenate([-b_f, [p_star - eps]]),
                    bounds=[(0, None)] * 2, method="highs")
    print(f"eps = {eps:6.3f}: 'beat p* by eps' feasible = {tight.status == 0}  "
          f"(status {tight.status})")
    assert tight.status != 0                      # infeasible for every eps > 0

dual = linprog(-b_f, A_ub=A_f.T, b_ub=c_f, bounds=[(0, None)] * 2, method="highs")
y_star = dual.x
print(f"\np* = {p_star:.6f}   d* = {-dual.fun:.6f}   gap = {abs(p_star + dual.fun):.2e}")
print(f"dual optimum y* = {y_star} is a vertex of "
      f"{{y >= 0 : A^T y <= c}}: active constraints = "
      f"{np.flatnonzero(np.abs(A_f.T @ y_star - c_f) < 1e-9).tolist()}")
assert abs(p_star + dual.fun) < 1e-9
assert (A_f.T @ y_star <= c_f + 1e-9).all() and (y_star >= -1e-9).all()

# Farkas on an outright infeasible system: {x >= 0 : Mx = q} with certificate w.
M_f = np.array([[1.0, 2.0], [2.0, 4.0]])
q_f = np.array([1.0, 3.0])
w_f = np.array([-2.0, 1.0])
print(f"\nFarkas: M^T w = {M_f.T @ w_f} (<= 0) and q^T w = {q_f @ w_f:.1f} (> 0) "
      f"-> {{x >= 0 : Mx = q}} is empty")
assert (M_f.T @ w_f <= 1e-12).all() and q_f @ w_f > 0

eps =  0.001: 'beat p* by eps' feasible = False  (status 2)
eps =  0.010: 'beat p* by eps' feasible = False  (status 2)
eps =  0.100: 'beat p* by eps' feasible = False  (status 2)

p* = 10.000000   d* = 10.000000   gap = 0.00e+00
dual optimum y* = [2. 1.] is a vertex of {y >= 0 : A^T y <= c}: active constraints = [0, 1]

Farkas: M^T w = [0. 0.] (<= 0) and q^T w = 1.0 (> 0) -> {x >= 0 : Mx = q} is empty


The system "achieve a cost below $p^\star$" is infeasible for every $\epsilon \gt 0$, which is what Step 2
feeds to Farkas; the dual attains $d^\star = p^\star = 10$ at a point where both dual constraints are
active, i.e. a vertex; and the last two lines display an explicit alternative-2 certificate for an
outright infeasible nonnegative system.